# HireMind · Reviewed continuation notebook

**Assignment 3 working research notebook; Assignment 4 remains separately gated.**

This version integrates the Teacher reporting fix and the previously executed storage-folder fix. Scientific training, preprocessing, seeds, hyperparameters, losses and quantization implementations are unchanged. Original trial records and target-cache provenance are never overwritten. The recovered Teacher was checked against all 4,993 original cached training logits: exact equality was reported, but the serialized file hashes differ. This is not a claim of byte-identical weights or equivalence on all possible inputs.

**Run now:** use a CPU runtime; execute **S01–S15** in order, leaving every `RUN_...` flag and `EXPORT_MODEL` false. Execute **R01** to verify artifacts and **C01** for the incident audit. CPU is sufficient for these checks; no training should start. If a dependency cell requests a restart, restart and rerun the definitions before R01. Do not use Run all. Send the new inventory output before proceeding to final-test gates.

The scientific outputs retained under R03, R05–R09 are from the uploaded Colab execution, not from execution of this edited notebook. New reporting changes were syntax-checked and tested with mocked artifacts locally; live Drive/model validation remains to be run in your Colab. The original uploaded notebook is the full incident archive.

**Still pending:** two independent human 90-pair label sheets, human error review, author verification and remote GitHub/Overleaf publication. No final test or deployment readiness is claimed. The A3 report uses validation results only. Do not retrain completed experiments to remove an old status label.


### Dependency repair — read before continuing
This revision fixes setup after package changes: conditional installation, an explicit restart gate, and an S03 prerequisite check for S04. The uploaded notebook showed an S03 NumPy import failure before the S04 NameError. Scientific functions, model settings, saved paths, and experiment algorithms are unchanged. Old cell outputs are cleared to avoid presenting them as new execution. Original outputs remain in your uploaded backup.
For Assignment 3 verification: use CPU, run S01–S15 in order (restart only if S02 requests it), leave execution flags False, then run R01 and C01 only. Do not use Run all. This file still contains separately labelled Assignment 4 sections.


# HireMind · Assignments 2, 3 and 4
### Rawan Khojah · Hams Aljohani | CS4085 Deep Learning
**One visible-code research notebook.** M0: supervised BERT-Tiny → M1: teacher-guided BERT-Tiny → M2: quantized M1.

This notebook continues the approved protocol. Historical Item 2 outputs are preserved separately. Item 3 and Item 4 are **not complete merely because their code is present**. Actual Drive-run status is unknown until R01 executes.

**Start here:** run S01–S15 in order (setup and definitions only), then R01. Follow the run guide below. Do not use Run all with final-test flags enabled. Main training logic is directly visible in cells; no encoded implementation or external training scripts.

| Milestone | Reused work | Work still requiring evidence |
|---|---|---|
| Item 2 | Original audit/splits, references, overfit check, M0, historical metrics | Optional reproduction is separate; don't overwrite historical results |
| Item 3 | Same M0/M1/M2 protocol, seeds, training budget | Validate saved trials; finish missing training, ablations, fair profiling, tables and interpretation |
| Item 4 | Valid checkpoints and Item 3 measurements | Finish planned experiments, final test after freeze (from our proposal), error analysis, full paper, presentation, poster and reproducibility evidence |

**Assignment 4 specifies:** final paper 8%; oral presentation/poster 2%; code/reproducibility 2%. Team presentation 12–15 minutes, every member speaks; poster A1/A0 with GitHub QR. Final ZIP: `CS4085_Project_GroupX_TopicY.zip`, containing paper PDF, slides PDF/PPTX, poster PDF, README links and AI log if separate. Replace GroupX/TopicY with your real identifiers. This notebook is the single code artifact, not the entire final submission.

**Research unchanged:** 256 combined tokens, original frozen split, AdamW/FP32, three epochs/939 updates, four M0 trials, four M1 trials, temperatures 2/4, selected seeds 42/43/44, T=1 ablation, Linear/embedding/both ablation. No automatic 512-token change or substitute method.

**Engineering changes:** visible functions; read-only legacy reuse; full recovery checkpoints; session-specific CPU benchmarks; gated final-test evaluation; portable CPU export. Old interrupted trials have no full recovery state and restart only that incomplete trial. Future interruptions resume from the latest valid saved update (up to 99 unsaved updates may repeat). Two rolling full snapshots per new trial bound disk use; legacy files and completed weights/logs are preserved.


## S01 · Mount Drive and locate the old run
Run this first. It reads the old run and uses a separate continuation folder. It does not delete or overwrite old files. Do not run two training notebooks against the same continuation folder simultaneously.

In [ ]:
from pathlib import Path
import os, sys, json
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive')
else:
    BASE = Path('/content')
LEGACY = BASE / 'HireMind_Item3_Run'
WORK = BASE / 'HireMind_Continuation_v2'
ROOT = WORK / 'research_assets'
ROOT.mkdir(parents=True, exist_ok=True)
old_record = LEGACY / 'public/run_configuration.json'
old_runtime = json.loads(old_record.read_text())['environment'] if old_record.exists() else {}
print('Legacy folder (read-only):', LEGACY)
print('Continuation folder:', WORK)
print('Recorded legacy runtime:', old_runtime or 'Not found; completion is not assumed.')


## S02 · Dependencies and session restart
Run S01 first. This cell installs only when required package versions differ. If it prints STOP, select **Runtime → Restart session**, then run S01 and S02 again before S03. Re-running S02 alone does not clear imported libraries. No PyTorch replacement or training is performed.


In [ ]:
import subprocess
packages = {
    'transformers': old_runtime.get('transformers', '4.51.3'),
    'numpy': old_runtime.get('numpy', '2.1.3'),
    'pandas': old_runtime.get('pandas', '2.2.3'),
    'huggingface-hub': '0.30.2',
    'scikit-learn': '1.6.1',
    'matplotlib': '3.10.1',
}
from importlib import metadata

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

changes = {name: version for name, version in packages.items()
           if installed_version(name) != version}
if changes:
    print('Installing required versions:', changes)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    *[f'{name}=={version}' for name, version in packages.items()]], check=True)
    DEPENDENCIES_REQUIRE_RESTART = True
    print('STOP: use Runtime > Restart session, then run S01, S02 and S03 again.')
    print('Do not continue to S03 in this session. Drive checkpoints are unchanged.')
else:
    print('Required dependency versions already installed; no installation performed.')
    if globals().get('DEPENDENCIES_REQUIRE_RESTART', False):
        print('STOP: restart the session before continuing; an earlier installation changed libraries.')
    else:
        print('Continue to S03. PyTorch has not been replaced.')


## S03 · Imports and unchanged scientific configuration
Continue only after S02 permits it. Wait for **S03 PASS** before S04. If any import fails, stop at that first error rather than running later cells.


In [ ]:
# Stop before importing a mixture of old in-memory and newly installed packages.
if globals().get('DEPENDENCIES_REQUIRE_RESTART', False):
    raise RuntimeError('Dependencies changed in this session. Use Runtime > Restart session, then run S01–S03 again.')
from importlib import metadata as _metadata
import sys as _sys
for _package, _module in [('numpy', 'numpy'), ('pandas', 'pandas'), ('transformers', 'transformers')]:
    _loaded = _sys.modules.get(_module)
    if _loaded is not None and getattr(_loaded, '__version__', None) != _metadata.version(_package):
        raise RuntimeError(f'{_package} in memory differs from its installed version. Restart the session and run S01–S03 again.')

"""HireMind Item 3. Validation only. Full runs require a GPU; smoke tests are not results."""
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import shutil, copy, gc, hashlib, json, math, platform, random, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertConfig,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from huggingface_hub import hf_hub_download, model_info
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset
from torch.ao.quantization import (
    quantize_dynamic,
    default_dynamic_qconfig,
    float_qparams_weight_only_qconfig,
)
LABELS = ["No Fit", "Potential Fit", "Good Fit"]
STUDENT = "google/bert_uncased_L-2_H-128_A-2"
STUDENT_REV = "30b0a37ccaaa32f332884b96992754e246e48c5f"
TEACHER = "google-bert/bert-base-uncased"
DATASET = "cnamuangtoun/resume-job-description-fit"
DATA_REV = "08978e21714984bb417547d2c0f9b477f5298163"
TRAIN_SHA = "b86fa488a21258015de1750633a9ac410f701ff96bf533f68978dbe3adec6d40"
SPLIT_SHA = "2ff3926c70152e6d5f47fbf40acebbb287e3311969618a8c7a20f45861539f9b"
CONFIG = dict(
    max_len=256,
    batch=16,
    epochs=3,
    eval_every=100,
    weight_decay=0.01,
    clip_norm=1.0,
    warmup_fraction=0.1,
    seeds=[42, 43, 44],
    m0_lrs=[2e-5, 5e-5, 1e-4, 3e-4],
    teacher_lrs=[2e-5, 5e-5],
    m1_lrs=[5e-5, 1e-4],
    temperatures=[2.0, 4.0],
    alpha=0.5,
    teacher_micro_batch=4,
    student_micro_batch=16,
    validation_batch=16,
    cpu_threads=1,
    benchmark_examples=20,
    warmup=30,
    timed=200,
    precision="float32",
    test_evaluation=False,
)
"""HireMind DL - dataset audit, cleaning and grouped train/validation/test split.

Pure pandas / scikit-learn (no PyTorch), so it can be tested anywhere.
Used by notebooks/HireMind_M0_Baseline.ipynb.

Split policy (fixed before any model training):
  * IDs: cv_id / job_id = first 16 hex chars of SHA-256 of whitespace-normalised, lower-cased text.
  * Near-duplicate job descriptions (TF-IDF cosine >= NEAR_DUP_THRESHOLD) share one job_group.
  * Every row of a pair that carries conflicting labels is removed (any split).
  * Exact duplicate pairs: one copy is kept; if a pair occurs in both files, the training copy is dropped.
  * Training rows whose job is a near-duplicate of a test job are dropped.
  * The official test file is kept as the test set (only the removals above).
  * Validation = one fold of StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    on the cleaned training file, grouped by job_group, stratified by label.
"""
import hashlib
import json
import re
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedGroupKFold
LABELS = ["No Fit", "Potential Fit", "Good Fit"]
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
NEAR_DUP_THRESHOLD = 0.95
from datetime import datetime, timezone
import uuid, inspect, textwrap, urllib.request
TEST_SHA = "f3c48f4cc3559bf192ebb9c241a30c601b5b8b3eee6f1656dd4d26dfd4b0887a"
LEGACY_CODE_HASHES = {'ec68e19379fabe6a629566c0d8f0c082408ec3f0e5c42a0a603fa5286f10faab'}

S03_READY = True
print('S03 PASS: imports and original scientific configuration loaded.')


## S04 · Fetch the frozen ID split and preserve historical results
Only ordinary text/CSV/JSON research assets are fetched, at a fixed GitHub commit. No implementation code is downloaded. Checksums prevent a changed split being used. The historical numbers below are copied from the supplied Item 2 result file, not newly generated results.

In [ ]:
if not globals().get('S03_READY', False) or 'SPLIT_SHA' not in globals():
    raise RuntimeError('S03 has not completed successfully. Fix its first error before running S04; do not bypass the split checksum.')
import urllib.request
GITHUB_COMMIT = 'c0b4eda2d0373d14098a03da4b59a041c1cf9860'
split_path = ROOT / 'splits/splits.csv'
split_path.parent.mkdir(parents=True, exist_ok=True)
if not split_path.exists():
    url = f'https://raw.githubusercontent.com/Hams770/HireMind-DL/{GITHUB_COMMIT}/splits/splits.csv'
    raw = urllib.request.urlopen(url, timeout=60).read()
    assert hashlib.sha256(raw).hexdigest() == SPLIT_SHA
    temporary = split_path.with_suffix('.download')
    temporary.write_bytes(raw)
    temporary.replace(split_path)
assert hashlib.sha256(split_path.read_bytes()).hexdigest() == SPLIT_SHA

HISTORICAL_ITEM2 = {'env': {'python': '3.13.15',
         'torch': '2.11.0+cu130',
         'transformers': '5.20.1',
         'sklearn': '1.6.1',
         'device': 'cuda',
         'gpu': 'Tesla T4',
         'numpy': '2.1.3',
         'pandas': '2.2.3',
         'huggingface_hub': '1.33.0',
         'student_model': 'google/bert_uncased_L-2_H-128_A-2',
         'student_revision': '30b0a37ccaaa32f332884b96992754e246e48c5f',
         'dataset_revision': '08978e21714984bb417547d2c0f9b477f5298163',
         'cpu': 'Intel(R) Xeon(R) CPU @ 2.00GHz',
         'cpu_count': 2},
 'references': {'majority': {'macro_f1': 22.171165996553707, 'accuracy': 49.827882960413085},
                'tfidf_logreg': {'macro_f1': 36.72115590468192, 'accuracy': 52.83993115318416}},
 'overfit': {'batch_size': 16,
             'steps': 300,
             'lr': 0.001,
             'first_loss': 1.1503040790557861,
             'final_loss': 0.0008686200599186122,
             'final_accuracy': 100.0,
             'passed': True},
 'lr_search': [{'lr': 2e-05,
                'best': {'loss': 1.0070846653230834,
                         'macro_f1': 25.844504215875315,
                         'accuracy': 50.94664371772806,
                         'step': 900,
                         'epoch': 3},
                'train_time_s': 26.71353816986084,
                'steps': 939,
                'peak_gpu_mem_mib': 406.25390625},
               {'lr': 5e-05,
                'best': {'loss': 0.9781261988227831,
                         'macro_f1': 41.960584976257515,
                         'accuracy': 53.27022375215146,
                         'step': 939,
                         'epoch': 3},
                'train_time_s': 17.998303174972534,
                'steps': 939,
                'peak_gpu_mem_mib': 406.25390625},
               {'lr': 0.0001,
                'best': {'loss': 0.9742252789758363,
                         'macro_f1': 46.536276922018175,
                         'accuracy': 55.59380378657487,
                         'step': 939,
                         'epoch': 3},
                'train_time_s': 18.052594423294067,
                'steps': 939,
                'peak_gpu_mem_mib': 406.25390625},
               {'lr': 0.0003,
                'best': {'loss': 1.1821041222078255,
                         'macro_f1': 41.4757349611974,
                         'accuracy': 52.323580034423415,
                         'step': 900,
                         'epoch': 3},
                'train_time_s': 18.186042070388794,
                'steps': 939,
                'peak_gpu_mem_mib': 406.25390625}],
 'selected_lr': 0.0001,
 'seeds': {'seeds': [42, 43, 44],
           'val_macro_f1': [46.536276922018175, 37.646805064438716, 35.85511233790764],
           'val_accuracy': [55.59380378657487, 51.54905335628227, 50.34423407917383],
           'macro_f1_mean': 40.01273144145485,
           'macro_f1_sd': 5.720142037948864,
           'accuracy_mean': 52.49569707401033,
           'accuracy_sd': 2.7498356813546407},
 'val_report': {'No Fit': {'precision': 0.6341789052069426,
                           'recall': 0.8203799654576857,
                           'f1-score': 0.7153614457831325,
                           'support': 579.0},
                'Potential Fit': {'precision': 0.3849056603773585,
                                  'recall': 0.3469387755102041,
                                  'f1-score': 0.36493738819320215,
                                  'support': 294.0},
                'Good Fit': {'precision': 0.46621621621621623,
                             'recall': 0.23875432525951557,
                             'f1-score': 0.3157894736842105,
                             'support': 289.0},
                'accuracy': 0.5559380378657487,
                'macro avg': {'precision': 0.4951002606001724,
                              'recall': 0.4686910220758018,
                              'f1-score': 0.46536276922018177,
                              'support': 1162.0},
                'weighted avg': {'precision': 0.529335918031196,
                                 'recall': 0.5559380378657487,
                                 'f1-score': 0.527322742798599,
                                 'support': 1162.0}},
 'val_confusion': [[475, 65, 39], [152, 102, 40], [122, 98, 69]],
 'cost': {'params': {'embeddings': 3972864,
                     'encoder.layer.0': 198272,
                     'encoder.layer.1': 198272,
                     'pooler': 16512,
                     'classifier': 387,
                     'total': 4386307,
                     'word_embeddings': 3906816},
          'macs_measured': {'embeddings': 0.0,
                            'encoder.layer.0': 67108864.0,
                            'encoder.layer.1': 67108864.0,
                            'pooler': 16384.0,
                            'classifier': 384.0,
                            'total': 134234496.0},
          'macs_analytic': {'per_layer': 67108864,
                            'pooler': 16384,
                            'classifier': 384,
                            'total': 134234496},
          'config': {'layers': 2,
                     'hidden': 128,
                     'heads': 2,
                     'intermediate': 512,
                     'vocab': 30522,
                     'max_len': 256},
          'serialized_bytes': 17558120,
          'cpu_bench': {'threads': 1,
                        'batch_size': 1,
                        'seq_len': 256,
                        'warmup': 30,
                        'runs': 200,
                        'max_chars': None,
                        'revision': '30b0a37ccaaa32f332884b96992754e246e48c5f',
                        'model_only': {'median_ms': 5.914109999991979, 'p95_ms': 6.765000000029886},
                        'tokenization_inclusive': {'median_ms': 25.471621000008327,
                                                   'p95_ms': 31.074773000000278},
                        'rss_after_imports_mib': 770.5546875,
                        'peak_rss_mib': 864.96484375,
                        'rss_end_mib': 864.96484375,
                        'torch': '2.11.0+cu130'}},
 'best_run_history': {'step': [100, 200, 300, 400, 500, 600, 700, 800, 900, 939],
                      'train_loss': [1.0622976595163345,
                                     1.0185886138677598,
                                     0.9941313254833222,
                                     0.9587893676757813,
                                     0.9332412284612656,
                                     0.8719972711801529,
                                     0.8156799983978271,
                                     0.769871027469635,
                                     0.7984420949220657,
                                     0.8092761391248459],
                      'val_loss': [1.0453790612967617,
                                   1.00931523465864,
                                   0.9949752396027111,
                                   0.982143642775163,
                                   0.9814528501792947,
                                   0.9743443965501506,
                                   0.9758370454052845,
                                   0.9798934666917575,
                                   0.9746542251869241,
                                   0.9742252789758363],
                      'val_macro_f1': [22.171165996553707,
                                       27.970316811360117,
                                       32.23073623143477,
                                       42.82488132384576,
                                       40.69191332995014,
                                       43.57563631878405,
                                       39.72091288440269,
                                       40.776879391180216,
                                       45.62561616367069,
                                       46.536276922018175],
                      'epoch_end_steps': [313, 626, 939]},
 'note': 'Validation results only; the test set was not evaluated.'}
print('Historical Item 2:', HISTORICAL_ITEM2['seeds'])


## S05 · Original Item 2 audit and split functions
Definitions only: running this cell does not start training.

In [ ]:
def norm(text):
    return re.sub(r"\s+", " ", str(text)).strip().lower()

def text_id(text):
    return hashlib.sha256(norm(text).encode("utf-8")).hexdigest()[:16]

def add_ids(train_df, test_df):
    """Return one frame with split, orig_index, cv_id, job_id, pair_id, label_id."""
    frames = []
    for name, df in (("train", train_df), ("test", test_df)):
        d = df[["resume_text", "job_description_text", "label"]].copy()
        d["orig_split"] = name
        d["orig_index"] = np.arange(len(d))
        frames.append(d)
    data = pd.concat(frames, ignore_index=True)
    unknown = set(data["label"]) - set(LABELS)
    if unknown:
        raise ValueError(f"Unexpected labels: {unknown}")
    data["cv_id"] = data["resume_text"].map(text_id)
    data["job_id"] = data["job_description_text"].map(text_id)
    data["pair_id"] = data["cv_id"] + "_" + data["job_id"]
    data["label_id"] = data["label"].map(LABEL2ID)
    data["row_uid"] = data["orig_split"] + "_" + data["orig_index"].astype(str)
    return data

def near_duplicate_groups(texts_by_id, threshold=NEAR_DUP_THRESHOLD):
    """Union-find over TF-IDF cosine similarity. Returns {id: group_id} and the list of near-dup pairs."""
    ids = list(texts_by_id.keys())
    if len(ids) < 2:
        return {i: i for i in ids}, []
    tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    matrix = tfidf.fit_transform([texts_by_id[i] for i in ids])
    sim = cosine_similarity(matrix)
    np.fill_diagonal(sim, 0.0)
    parent = list(range(len(ids)))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    pairs = []
    rows, cols = np.where(np.triu(sim, k=1) >= threshold)
    for r, c in zip(rows, cols):
        pairs.append((ids[r], ids[c], float(sim[r, c])))
        parent[find(r)] = find(c)
    groups = {ids[i]: ids[find(i)] for i in range(len(ids))}
    return groups, pairs

def audit(data):
    """Dataset statistics before cleaning."""
    stats = {"rows": int(len(data))}
    for split in ("train", "test"):
        d = data[data.orig_split == split]
        stats[split] = {
            "rows": int(len(d)),
            "class_counts": {k: int(v) for k, v in Counter(d["label"]).items()},
            "unique_cvs": int(d.cv_id.nunique()),
            "unique_jobs": int(d.job_id.nunique()),
        }
    stats["class_counts_total"] = {k: int(v) for k, v in Counter(data["label"]).items()}
    stats["unique_cvs"] = int(data.cv_id.nunique())
    stats["unique_jobs"] = int(data.job_id.nunique())
    labels_per_pair = data.groupby("pair_id")["label"].nunique()
    conflicting = labels_per_pair[labels_per_pair > 1].index
    stats["unique_pairs"] = int(data.pair_id.nunique())
    stats["duplicate_pair_excess_rows"] = int(len(data) - data.pair_id.nunique())
    stats["conflicting_pairs"] = int(len(conflicting))
    stats["conflicting_rows"] = int(data.pair_id.isin(conflicting).sum())
    tr, te = data[data.orig_split == "train"], data[data.orig_split == "test"]
    stats["overlap_train_test"] = {
        "cvs": int(len(set(tr.cv_id) & set(te.cv_id))),
        "jobs": int(len(set(tr.job_id) & set(te.job_id))),
        "pairs": int(len(set(tr.pair_id) & set(te.pair_id))),
    }
    return stats, set(conflicting)

def clean(data, conflicting_pairs):
    """Apply the removal rules; return cleaned frame and a removal log."""
    log = {}
    before = data.orig_split.value_counts().to_dict()
    d = data[~data.pair_id.isin(conflicting_pairs)].copy()
    log["removed_conflicting_rows"] = {
        s: int(before.get(s, 0) - (d.orig_split == s).sum()) for s in ("train", "test")
    }
    test_pairs = set(d.loc[d.orig_split == "test", "pair_id"])
    mask_cross = (d.orig_split == "train") & d.pair_id.isin(test_pairs)
    log["removed_train_rows_duplicated_in_test"] = int(mask_cross.sum())
    d = d[~mask_cross]
    n0 = d.orig_split.value_counts().to_dict()
    d = d.drop_duplicates(subset=["orig_split", "pair_id"], keep="first")
    log["removed_within_split_duplicates"] = {
        s: int(n0.get(s, 0) - (d.orig_split == s).sum()) for s in ("train", "test")
    }
    return d.reset_index(drop=True), log

def remove_train_rows_sharing_test_job_group(d, job_groups):
    """Drop training rows whose job is a near-duplicate of a test job (prevents job-level leakage)."""
    test_groups = {job_groups[j] for j in d.loc[d.orig_split == "test", "job_id"]}
    mask = (d.orig_split == "train") & d.job_id.map(job_groups).isin(test_groups)
    return d[~mask].reset_index(drop=True), int(mask.sum())

def make_splits(d, job_groups, seed=42, n_splits=5):
    """Grouped, stratified validation split from the cleaned training file."""
    d = d.copy()
    d["job_group"] = d["job_id"].map(job_groups)
    tr = d[d.orig_split == "train"]
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    _, val_idx = next(sgkf.split(tr, tr["label_id"], groups=tr["job_group"]))
    val_rows = set(tr.index[val_idx])
    d["split"] = np.where(
        d.orig_split == "test", "test", np.where(d.index.isin(val_rows), "validation", "train")
    )
    return d

def check_splits(d):
    """Assertions that the split policy holds. Returns a dict of checks."""
    s = {k: d[d.split == k] for k in ("train", "validation", "test")}
    checks = {
        "train_val_job_group_overlap": len(
            set(s["train"].job_group) & set(s["validation"].job_group)
        ),
        "train_test_job_overlap": len(set(s["train"].job_id) & set(s["test"].job_id)),
        "val_test_job_overlap": len(set(s["validation"].job_id) & set(s["test"].job_id)),
        "pair_overlap_any": len(set(s["train"].pair_id) & set(s["validation"].pair_id))
        + len(set(s["train"].pair_id) & set(s["test"].pair_id))
        + len(set(s["validation"].pair_id) & set(s["test"].pair_id)),
        "classes_in_validation": int(s["validation"].label_id.nunique()),
        "cv_overlap_train_test": len(set(s["train"].cv_id) & set(s["test"].cv_id)),
        "cv_overlap_train_val": len(set(s["train"].cv_id) & set(s["validation"].cv_id)),
    }
    assert (
        checks["train_val_job_group_overlap"] == 0
    ), "job group leaks between train and validation"
    assert checks["pair_overlap_any"] == 0, "identical pair in two splits"
    assert checks["classes_in_validation"] == 3, "validation is missing a class"
    return checks

def split_counts(d):
    out = {}
    for k in ("train", "validation", "test"):
        part = d[d.split == k]
        out[k] = {
            "rows": int(len(part)),
            "cvs": int(part.cv_id.nunique()),
            "jobs": int(part.job_id.nunique()),
            "class_counts": {LABELS[i]: int((part.label_id == i).sum()) for i in range(3)},
        }
    return out

def save_split_definition(d, path):
    """Save IDs only (no CV or job text is written)."""
    cols = [
        "row_uid",
        "orig_split",
        "orig_index",
        "split",
        "cv_id",
        "job_id",
        "job_group",
        "pair_id",
        "label",
        "label_id",
    ]
    d[cols].to_csv(path, index=False)

## S06 · Integrity, configuration and recovery infrastructure
Definitions only: running this cell does not start training.

In [ ]:
def dump(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(obj, indent=2, default=lambda x: x.item() if hasattr(x, "item") else str(x))
    )
    tmp.replace(path)

def read(path):
    return json.loads(Path(path).read_text())

def seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def environment():
    cpu = "unknown"
    if Path("/proc/cpuinfo").exists():
        cpu = next(
            (
                s.split(":", 1)[1].strip()
                for s in Path("/proc/cpuinfo").read_text().splitlines()
                if s.startswith("model name")
            ),
            cpu,
        )
    return dict(
        python=platform.python_version(),
        torch=torch.__version__,
        transformers=transformers.__version__,
        numpy=np.__version__,
        pandas=pd.__version__,
        platform=platform.platform(),
        cpu=cpu,
        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        cuda=torch.version.cuda,
    )

def paths(work):
    w = Path(work)
    p = w / "public"
    v = w / "private"
    p.mkdir(parents=True, exist_ok=True)
    v.mkdir(parents=True, exist_ok=True)
    return w, p, v

def utc():
    return datetime.now(timezone.utc).isoformat()

def json_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def atomic_save(obj, path):
    # Cloud verification is mandatory for new tensor writes in Drive.
    return verified_tensor_save(obj, path)

def frozen_write(path, obj):
    path = Path(path)
    if path.exists():
        if read(path) != obj:
            raise RuntimeError(f"Existing record differs; preserved unchanged: {path}")
    else:
        dump(path, obj)

def protocol():
    return dict(
        config=CONFIG,
        student=STUDENT,
        student_revision=STUDENT_REV,
        dataset=DATASET,
        dataset_revision=DATA_REV,
        split_sha256=SPLIT_SHA,
    )

def visible_source(obj):
    """Read the displayed definition; fall back to notebook history if linecache is unavailable."""
    import ast

    try:
        return textwrap.dedent(inspect.getsource(obj))
    except (OSError, TypeError):
        history = globals().get("In", [])
        for cell in reversed(history):
            try:
                tree = ast.parse(cell)
            except SyntaxError:
                continue
            for node in tree.body:
                if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name == obj.__name__:
                    start = min([node.lineno] + [d.lineno for d in node.decorator_list]) - 1
                    return "\n".join(cell.splitlines()[start : node.end_lineno])
        raise RuntimeError("Cannot recover visible source. Re-run the definition cell.")

def setup(work):
    w, p, v = paths(work)
    if sha(ROOT / "splits/splits.csv") != SPLIT_SHA:
        raise ValueError("Frozen split checksum differs.")
    frozen_write(p / "protocol.json", protocol())
    import ast

    reviewed_functions = ["prepare", "model", "evaluate", "kd_loss", "train_trial", "quantize"]
    implementation = {
        name: hashlib.sha256(
            ast.dump(ast.parse(visible_source(globals()[name])), include_attributes=False).encode()
        ).hexdigest()
        for name in reviewed_functions
    }
    original_file = p / 'implementation.json'
    approved_before_repair = {'prepare': '132c438a7bcdfdcc89588d63ea420a9d299e509001967f6325f91570c5d461ea', 'model': '54741370eb84623c272d90317eac169807b2235d204030fdba01593c14828553', 'evaluate': 'afea8f1b309aa80f72f89bccdb736f8ee375efa9caf371be4fb060056eab3aaf', 'kd_loss': 'f290214d731c0af4a11dd5f516e1d171fbbd20a19657364d068d4ecd04089dbc', 'train_trial': '873330a65db270169bb6805ec9a815acaf2ec0cc6457c35e23687b67e1a772b5', 'quantize': '643cc874eec9fe786e59230c0bcd43980727bcc3e4770a7bce23a39ca304eaae'}
    if original_file.exists():
        saved = read(original_file)
        if saved not in (approved_before_repair, implementation):
            raise RuntimeError('Unreviewed implementation fingerprint; no automatic migration.')
        frozen_write(p / 'implementation_recovery_v3.json', implementation)
    else:
        frozen_write(original_file, implementation)
    support_names = ['atomic_save', 'save_recovery', 'verified_tensor_save', 'upload_verified',
                     'verified_teacher_targets', 'training_environment_guard', 'compatible_runtime']
    support_hashes = {name: hashlib.sha256(ast.dump(ast.parse(visible_source(globals()[name]))).encode()).hexdigest()
                      for name in support_names}
    frozen_write(p / 'recovery_support_v3.json', support_hashes)
    # Keep first runtime and all subsequent session records; CPU evaluation may use a new runtime.
    env = environment()
    frozen_write(p / "sessions" / f"{json_hash(env)}.json", env)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    return w, p, v

def checked_legacy():
    f = LEGACY / "public/run_configuration.json"
    if not f.exists():
        return None
    old = read(f)
    for k, value in protocol().items():
        if old.get(k) != value:
            raise RuntimeError(f"Legacy scientific setting differs: {k}; no automatic reuse.")
    if old.get("software_sha256") not in LEGACY_CODE_HASHES:
        raise RuntimeError(
            "Legacy code hash is not the reviewed Item 3 runner; inspect before reuse."
        )
    return old

def training_environment_guard(work):
    current = environment()
    legacy = checked_legacy()
    f = Path(work) / 'public/training_environment.json'
    expected = read(f) if f.exists() else (legacy['environment'] if legacy else current)
    if not compatible_runtime(expected, current):
        raise RuntimeError('Training runtime differs; no silent bypass. ' + str(runtime_differences(expected, current)))
    frozen_write(f, expected)
    if runtime_differences(expected, current):
        frozen_write(Path(work) / 'public/recovery_incident/runtime_exceptions' / (json_hash(current) + '.json'),
                     dict(expected=expected, actual=current,
                          exception='Explicit Python patch-only continuation; bitwise equivalence is not established.'))
    return current

def tag_for(kind, lr, s, t=None):
    return f"{kind}_s{s}_lr{lr:g}" + (f"_T{t:g}" if t else "")

def trial_contract(kind, lr, s, t=None):
    return dict(protocol=protocol(), kind=kind, lr=lr, seed=s, temperature=t)

def checkpoint_path(root, info):
    path = Path(info["weights"])
    path = path if path.is_absolute() else Path(root) / path
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

def validate_trial(root, info, expected_tag=None):
    if expected_tag and info.get("tag") != expected_tag:
        raise ValueError("Trial tag mismatch.")
    if info.get("steps") != 939:
        raise ValueError("Incomplete update budget; not a completed trial.")
    if not 0 <= float(info["best"]["macro_f1"]) <= 100:
        raise ValueError("Invalid metric.")
    for key, value in [("effective_batch", 16), ("validation_batch", 16), ("precision", "float32")]:
        if info.get(key) != value:
            raise ValueError(f"Unexpected {key}.")
    file = checkpoint_path(root, info)
    if sha(file) != info["weights_sha256"]:
        raise ValueError("Checkpoint SHA256 mismatch.")
    state = torch.load(file, map_location="cpu", weights_only=True)
    assert info["kind"] in ["M0", "M1", "teacher"], "Unknown model kind."
    expected_hidden = 768 if info["kind"] == "teacher" else 128
    assert state["classifier.weight"].shape == (3, expected_hidden)
    assert state["bert.embeddings.word_embeddings.weight"].shape == (30522, expected_hidden)
    assert all(
        torch.isfinite(x).all()
        for x in state.values()
        if torch.is_tensor(x) and x.is_floating_point()
    )
    cfg = BertConfig(
        vocab_size=30522,
        hidden_size=expected_hidden,
        num_hidden_layers=12 if info["kind"] == "teacher" else 2,
        num_attention_heads=12 if info["kind"] == "teacher" else 2,
        intermediate_size=3072 if info["kind"] == "teacher" else 512,
        num_labels=3,
    )
    with torch.device("meta"):
        schema = BertForSequenceClassification(cfg).state_dict()
    assert set(state) == set(schema), "Checkpoint model keys differ."
    assert all(
        state[k].shape == schema[k].shape for k in schema
    ), "Checkpoint tensor shapes differ."
    out = {**info, "weights": str(file.resolve()), "source_root": str(Path(root).resolve())}
    del state
    return out

def reviewed_trial(root, info, expected_tag=None):
    """Validate an original trial or its explicitly evidenced recovered Teacher.

    Never edits historical trial records, cached targets, or checkpoint hashes.
    A corrupt existing original file is an error, not a reason to substitute it.
    The equivalence check covers the 4,993 cached training inputs only.
    """
    root = Path(root)
    if expected_tag and info.get('tag') != expected_tag:
        raise ValueError('Trial tag mismatch.')
    original_path = Path(info['weights'])
    if not original_path.is_absolute():
        original_path = root / original_path
    if original_path.is_file():
        return validate_trial(root, info, expected_tag)
    recovery_dir = root / 'public/recovery_incident/teacher_reconstruction'
    recovered_file = recovery_dir / 'teacher_recovered.json'
    equivalence_file = recovery_dir / 'teacher_equivalence.json'
    if info.get('kind') != 'teacher' or not recovered_file.is_file() or not equivalence_file.is_file():
        return validate_trial(root, info, expected_tag)  # Preserve the original failure.
    recovered = read(recovered_file)
    evidence = read(equivalence_file)
    for key in ['tag', 'kind', 'lr', 'seed', 'temperature', 'steps',
                'effective_batch', 'validation_batch', 'precision', 'best', 'contract']:
        if recovered.get(key) != info.get(key):
            raise ValueError('Recovered Teacher differs from original record: ' + key)
    if read(root / 'public/protocol.json') != protocol():
        raise ValueError('Recovered Teacher protocol mismatch.')
    binding = verified_teacher_targets(root, pin=False)
    required = {
        'teacher_tag': info['tag'],
        'original_weights_sha256': info['weights_sha256'],
        'recovered_weights_sha256': recovered['weights_sha256'],
        'cached_targets_sha256': binding['targets_sha256'],
        'examples_checked': 4993,
        'logits_exactly_equal': True,
        'maximum_absolute_difference': 0.0,
        'predicted_class_agreement_percent': 100.0,
        'original_targets_preserved': True,
    }
    for key, value in required.items():
        if evidence.get(key) != value:
            raise ValueError('Recovered Teacher evidence mismatch: ' + key)
    if binding['teacher_tag'] != info['tag'] or binding['teacher_weights_sha256'] != info['weights_sha256']:
        raise ValueError('Recovered Teacher target provenance mismatch.')
    verified = validate_trial(root, recovered, info['tag'])
    evidence_path = Path(evidence['recovered_weights'])
    if not evidence_path.is_absolute():
        evidence_path = root / evidence_path
    if evidence_path.resolve() != Path(verified['weights']).resolve():
        raise ValueError('Recovered Teacher evidence references another file.')
    return {**verified, 'recovery_verified': True,
            'original_weights_sha256': info['weights_sha256'],
            'recovery_evidence': str(equivalence_file),
            'equivalence_scope': evidence.get('scope', 'Original cached training inputs only')}

def completed_trial(work, kind, lr, s, t=None):
    tag = tag_for(kind, lr, s, t)
    for root in [Path(work), LEGACY]:
        f = root / "public/trials" / f"{tag}.json"
        if not f.exists():
            continue
        if root == LEGACY:
            if checked_legacy() is None:
                raise RuntimeError(
                    "Legacy trial has no run configuration; cannot verify compatibility."
                )
        else:
            if read(root / "public/protocol.json") != protocol():
                raise ValueError("Protocol mismatch.")
        info = read(f)
        for key, val in [("kind", kind), ("lr", lr), ("seed", s), ("temperature", t)]:
            if info.get(key) != val:
                raise ValueError(f"{tag}: {key} mismatch")
        verified = reviewed_trial(root, info, tag)
        if root == LEGACY:
            frozen_write(Path(work) / "public/adopted_trials" / f"{tag}.json", verified)
        return verified
    return None

def recovery_candidates(work, tag):
    folder = Path(work) / "private/recovery" / tag
    valid = []
    for meta in folder.glob("step_*.json"):
        try:
            rec = read(meta)
            f = folder / rec["file"]
            if f.is_file() and sha(f) == rec["sha256"]:
                valid.append((rec["step"], f, meta))
        except Exception as ex:
            print("Invalid recovery record retained:", meta.name, str(ex))
    return sorted(valid, key=lambda x: x[0], reverse=True)

def capture_rng():
    return dict(
        python=random.getstate(),
        numpy=np.random.get_state(),
        cpu=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    )

def restore_rng(r):
    random.setstate(r["python"])
    np.random.set_state(r["numpy"])
    torch.set_rng_state(r["cpu"])
    if r["cuda"] is not None:
        torch.cuda.set_rng_state_all(r["cuda"])

def save_recovery(work, tag, state):
    if not STORAGE_READY:
        raise RuntimeError('Storage probe required before saving recovery state.')
    folder = Path(work) / 'private/recovery' / tag
    folder.mkdir(parents=True, exist_ok=True)
    # Fixed two-slot rotation; old v2 snapshots and Bin are never deleted here.
    valid = recovery_candidates(work, tag)
    latest = valid[0][1].name if valid else None
    slot = 'B' if latest == 'slot_A.pt' else 'A'
    target = folder / f'slot_{slot}.pt'
    receipt = verified_tensor_save(state, target, overwrite=True)
    persist_recovery_metadata(folder / f'step_slot_{slot}.json',
         dict(file=target.name, sha256=sha(target), step=state['step'],
              created=utc(), cloud_file_id=receipt.get('id'),
              cloud_md5=receipt.get('md5Checksum')))
    print('Cloud-verified recovery:', tag, 'update', state['step'], 'slot', slot, flush=True)

def status_inventory(work):
    rows = []
    for root in [LEGACY, Path(work)]:
        for f in sorted((root / "public/trials").glob("*.json")):
            try:
                if root == LEGACY and checked_legacy() is None:
                    raise RuntimeError("Missing legacy run configuration")
                info = reviewed_trial(root, read(f), f.stem)
                status = ("COMPLETE RECOVERED: weights and original-target equivalence evidence validated"
                          if info.get("recovery_verified") else "COMPLETE: checksum and weights validated")
            except Exception as ex:
                status = "BLOCKED: " + str(ex)
            rows.append(
                dict(
                    origin="legacy" if root == LEGACY else "continuation",
                    trial=f.stem,
                    status=status,
                )
            )
        known_weights = set()
        for record in (root / "public/trials").glob("*.json"):
            try:
                known_weights.add(Path(reviewed_trial(root, read(record), record.stem)["weights"]).resolve())
            except Exception:
                pass
        for f in (root / "private/checkpoints").glob("*.pt"):
            if f.resolve() not in known_weights and not f.stem.startswith("Q_"):
                rows.append(
                    dict(
                        origin=str(root),
                        trial=f.stem,
                        status="WEIGHTS ONLY: not proof of completion; old trial cannot resume optimizer",
                    )
                )
    for folder in (Path(work) / "private/recovery").glob("*"):
        valid = recovery_candidates(work, folder.name)
        if valid:
            rows.append(
                dict(
                    origin="continuation",
                    trial=folder.name,
                    status=f"RECOVERY: last valid update {valid[0][0]} (runtime/contract checked when resumed)",
                )
            )
    df = pd.DataFrame(rows, columns=["origin", "trial", "status"])
    print(
        df.to_string(index=False)
        if len(df)
        else "No completed/recoverable trials found. No training has been started by this check."
    )
    p = Path(work) / "public"
    p.mkdir(parents=True, exist_ok=True)
    dump(p / "inventory" / f"{uuid.uuid4().hex}.json", dict(checked_at=utc(), rows=rows))
    return df
# Visible recovery support: infrastructure only; research settings remain unchanged.
DRIVE_SERVICE = None
STORAGE_READY = False
ALLOW_PYTHON_PATCH_CHANGE = False
RESTART_MISSING_TRIALS = set()  # Populate only after reviewing the incident audit.
RESTORED_TRIALS = set()         # Trials whose restored snapshots may resume.


def runtime_differences(expected, actual):
    keys = ['python', 'torch', 'transformers', 'numpy', 'pandas', 'cuda', 'gpu']
    return {k: [expected.get(k), actual.get(k)] for k in keys
            if expected.get(k) != actual.get(k)}


def compatible_runtime(expected, actual):
    differences = runtime_differences(expected, actual)
    if not differences:
        return True
    # Optional, explicitly disclosed Python patch-only exception. Never change CUDA/PyTorch/GPU.
    return (ALLOW_PYTHON_PATCH_CHANGE and set(differences) == {'python'}
            and expected['python'].split('.')[:2] == actual['python'].split('.')[:2])


def authenticate_storage():
    global DRIVE_SERVICE
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    DRIVE_SERVICE = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    account = DRIVE_SERVICE.about().get(fields='user(emailAddress),storageQuota').execute()
    print('Drive API account:', account['user']['emailAddress'])
    print('Use the SAME account as the mounted project folders.')
    quota = account['storageQuota']
    if 'limit' in quota:
        free = int(quota['limit']) - int(quota.get('usage', 0))
        print('Cloud storage free (GiB):', round(free / 2**30, 2))
    else:
        print('No individual storage limit returned; organizational limits may apply.')
    return quota


def drive_children(parent_id, name):
    def escape(value):
        return value.replace('\\', '\\\\').replace("'", "\\'")
    query = f"'{escape(parent_id)}' in parents and name = '{escape(name)}' and trashed = false"
    rows, token = [], None
    while True:
        result = DRIVE_SERVICE.files().list(
            q=query, spaces='drive', fields='nextPageToken,files(id,name,mimeType,size,md5Checksum)',
            pageSize=100, pageToken=token).execute()
        rows.extend(result.get('files', []))
        token = result.get('nextPageToken')
        if not token:
            break
    if len(rows) > 1:
        raise RuntimeError(f'Ambiguous duplicate Drive name: {name}. No file changed.')
    return rows


def drive_parent(path):
    """Create folders through mounted Drive only; resolve IDs through API."""
    path = Path(path)
    work = Path(WORK)
    path.relative_to(work)

    mount_root = Path("/content/drive/MyDrive")
    parts = path.parent.relative_to(mount_root).parts
    parent_id = "root"
    mounted_folder = mount_root

    for name in parts:
        mounted_folder = mounted_folder / name
        rows = drive_children(parent_id, name)

        if not rows:
            # Never recreate the project root or its ancestors.
            if mounted_folder == work or work not in mounted_folder.parents:
                raise RuntimeError(
                    f"Project folder not found through API: {mounted_folder}"
                )

            # One folder-creation mechanism only.
            mounted_folder.mkdir(exist_ok=True)

            for _ in range(30):
                rows = drive_children(parent_id, name)
                if rows:
                    break
                time.sleep(2)

            if not rows:
                raise RuntimeError(
                    f"Folder synchronization pending: {mounted_folder}. "
                    "Stopped without creating an API duplicate."
                )

        if len(rows) != 1:
            raise RuntimeError(f"Ambiguous cloud folder: {mounted_folder}; review duplicate folders before writing.")

        if rows[0]["mimeType"] != "application/vnd.google-apps.folder":
            raise RuntimeError(f"Expected a folder: {mounted_folder}")

        parent_id = rows[0]["id"]

    return parent_id


def upload_verified(local_file, destination, overwrite=False):
    """Upload bytes to Drive and verify the server checksum; never permanently delete files."""
    if DRIVE_SERVICE is None:
        raise RuntimeError('Run C02 storage connection/probe before new tensor saves.')
    from googleapiclient.http import MediaFileUpload
    local_file, destination = Path(local_file), Path(destination)
    destination.relative_to(Path(WORK))
    checksum = hashlib.md5()
    with local_file.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b''):
            checksum.update(chunk)
    digest = checksum.hexdigest()
    parent = drive_parent(destination)
    rows = drive_children(parent, destination.name)
    if rows and rows[0].get('md5Checksum') == digest:
        return rows[0]
    if rows and not overwrite:
        raise RuntimeError(f'Existing file differs; preserved: {destination}')
    quota = DRIVE_SERVICE.about().get(fields='storageQuota').execute()['storageQuota']
    # Conservative: allow a whole new upload plus 1 GiB reserve, even for a slot update.
    needed = local_file.stat().st_size + 2**30
    if 'limit' in quota and int(quota['limit']) - int(quota.get('usage', 0)) < needed:
        raise RuntimeError('Insufficient cloud storage headroom. Local staging file retained.')
    media = MediaFileUpload(str(local_file), mimetype='application/octet-stream',
                            chunksize=16 * 1024**2, resumable=True)
    fields = 'id,name,size,md5Checksum'
    if rows:
        request = DRIVE_SERVICE.files().update(fileId=rows[0]['id'], media_body=media,
                                               fields=fields)
    else:
        request = DRIVE_SERVICE.files().create(
            body={'name': destination.name, 'parents': [parent]}, media_body=media, fields=fields)
    response = None
    while response is None:
        _, response = request.next_chunk(num_retries=3)
    receipt = DRIVE_SERVICE.files().get(fileId=response['id'], fields=fields).execute()
    if receipt.get('md5Checksum') != digest or int(receipt.get('size', -1)) != local_file.stat().st_size:
        raise RuntimeError('Cloud byte verification failed. Local staging file retained.')
    # A mounted Drive directory may lag behind its API. Stop, rather than create duplicate files.
    for _ in range(15):
        if destination.is_file() and sha(destination) == sha(local_file):
            return receipt
        time.sleep(2)
    raise RuntimeError('Cloud upload verified, but mount has not refreshed. Do NOT retrain. '
                       'Remount Drive and inspect the saved file. Local staging file retained.')


def verified_tensor_save(obj, path, overwrite=False):
    import tempfile
    path = Path(path)
    staging = Path('/content/hiremind_upload_staging')
    staging.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=staging, suffix='.pt', delete=False) as stream:
        local = Path(stream.name)
        torch.save(obj, stream)
        stream.flush()
        os.fsync(stream.fileno())
    if str(path).startswith('/content/drive/'):
        receipt = upload_verified(local, path, overwrite=overwrite)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists() and not overwrite:
            if sha(path) != sha(local):
                raise RuntimeError('Existing local tensor file differs; preserved.')
        else:
            shutil.copyfile(local, path)
        receipt = {'local_only': True}
    if sha(local) != sha(path):
        raise RuntimeError('Saved tensor checksum differs; staging file retained.')
    local.unlink()  # Only temporary local disk; never deletes Drive artifacts.
    return receipt


def storage_probe():
    global STORAGE_READY
    STORAGE_READY = False
    authenticate_storage()
    probe = Path(WORK) / 'private/storage_checks' / f'probe_{uuid.uuid4().hex}.pt'
    value = {'probe': torch.arange(1024, dtype=torch.int64)}
    verified_tensor_save(value, probe)
    loaded = torch.load(probe, map_location='cpu', weights_only=True)
    assert torch.equal(loaded['probe'], value['probe'])
    STORAGE_READY = True
    print('PASS: cloud upload, server checksum, mounted readback and tensor reload.')
    print('No old checkpoints were deleted. Training is still controlled by RUN_TRAINING.')


def verified_teacher_targets(work, pin=False):
    """Validate the ORIGINAL cached targets; never regenerate them with a different Teacher."""
    w = Path(work)
    selected = read(w / 'public/teacher_selected.json')
    trial_file = w / 'public/trials' / (selected['tag'] + '.json')
    if not trial_file.exists():
        trial_file = LEGACY / 'public/trials' / (selected['tag'] + '.json')
    trial = read(trial_file)
    assert selected['kind'] == 'teacher' and selected['steps'] == trial['steps'] == 939
    assert trial['weights_sha256'] == selected['weights_sha256']
    assert selected['best'] == trial['best']
    assert read(w / 'public/protocol.json') == protocol()
    path = w / 'private/teacher_train_logits.pt'
    cache = torch.load(path, map_location='cpu', weights_only=True)
    ids = read(w / 'public/data_used.json')['training_row_ids']
    assert cache['training_row_ids'] == ids and len(ids) == 4993
    assert cache['teacher_weights_sha256'] == selected['weights_sha256']
    assert tuple(cache['logits'].shape) == (4993, 3)
    assert cache['logits'].dtype == torch.float32
    assert torch.isfinite(cache['logits']).all()
    provenance = dict(teacher_tag=selected['tag'], teacher_weights_sha256=selected['weights_sha256'],
                      targets_sha256=sha(path), training_ids_sha256=json_hash(ids))
    binding = w / 'public/recovery_incident/original_teacher_targets.json'
    if binding.exists():
        assert read(binding) == provenance, 'Original Teacher targets changed; stop.'
    elif pin:
        frozen_write(binding, provenance)
    del cache
    return provenance


def recovery_audit(work):
    """Read-only inventory; training logs alone never count as recoverable weights."""
    w = Path(work)
    print('Current environment:', environment())
    expected_file = w / 'public/training_environment.json'
    if expected_file.exists():
        print('Training environment differences:', runtime_differences(read(expected_file), environment()))
    try:
        binding = verified_teacher_targets(work, pin=False)
        print('ORIGINAL TEACHER TARGETS: validated against saved trial, labels/row order and shape.')
        print('Teacher tag:', binding['teacher_tag'])
        teacher_info = reviewed_trial(w, read(w / 'public/teacher_selected.json'))
        print('TEACHER WEIGHTS: recovered and verified.' if teacher_info.get('recovery_verified') else 'TEACHER WEIGHTS: original file verified.')
    except Exception as error:
        print('TEACHER TARGETS BLOCKED:', type(error).__name__, str(error))
    tags = {tag_for('M0', lr, 42) for lr in CONFIG['m0_lrs']}
    tags |= {tag_for('teacher', lr, 42) for lr in CONFIG['teacher_lrs']}
    tags |= {tag_for('M1', lr, 42, t) for lr in CONFIG['m1_lrs'] for t in CONFIG['temperatures']}
    for file in (w / 'public').glob('*selected.json'):
        record = read(file)
        records = record if isinstance(record, list) else [record]
        tags |= {x['tag'] for x in records if 'tag' in x}
    tags |= set(INCIDENT_M1_TAGS)
    tags |= {x.stem for x in (w / 'public/trials').glob('*.json')}
    for folder in (w / 'private/recovery').glob('*'):
        if folder.is_dir(): tags.add(folder.name)
    rows = []
    for tag in sorted(tags):
        found = False
        for root in [w, LEGACY]:
            file = root / 'public/trials' / (tag + '.json')
            if not file.exists(): continue
            try:
                if root == LEGACY: checked_legacy()
                info = reviewed_trial(root, read(file), tag)
                rows.append(dict(trial=tag, status=('COMPLETE RECOVERED: weights and target-equivalence evidence verified' if info.get('recovery_verified') else 'COMPLETE: weights verified'), recovery_step=None))
                found = True
                break
            except Exception as error:
                rows.append(dict(trial=tag, status='RECORD BLOCKED: ' + str(error), recovery_step=None))
        if found: continue
        candidates = recovery_candidates(w, tag)
        if candidates:
            rows.append(dict(trial=tag, status='SNAPSHOT: checksum verified; inspect state before resume',
                             recovery_step=candidates[0][0]))
        elif not any(x['trial'] == tag for x in rows):
            rows.append(dict(trial=tag, status='NO COMPLETE RECORD OR RECOVERY', recovery_step=None))
    table = pd.DataFrame(rows)
    print(table.to_string(index=False))
    print('Nothing was trained, moved, deleted, or restored by C01.')
    return table


def authorize_trial_repair(work, tag, allow_restart=False):
    """Explicit, logged repair for one reviewed student trial; Teacher identity stays frozen."""
    if not STORAGE_READY:
        raise RuntimeError('Run C02 before authorizing a repair.')
    if not tag.startswith('M1_'):
        raise RuntimeError('This incident repair is limited to M1. Teacher retraining is not implicit.')
    verified_teacher_targets(work, pin=True)
    w = Path(work)
    source = w / 'public/trials' / (tag + '.json')
    if source.exists():
        info = read(source)
        try:
            validate_trial(w, info, tag)
            print('Already valid; no repair needed:', tag)
            return
        except FileNotFoundError:
            pass  # Only missing weights are eligible. Corruption/configuration errors stay blocked.
    candidates = recovery_candidates(w, tag)
    if candidates:
        r = torch.load(candidates[0][1], map_location='cpu', weights_only=False)
        assert r['recovery_version'] == 2 and r['best_state'] is not None
        contract = r['contract']
        assert contract['protocol'] == protocol() and contract['kind'] == 'M1'
        assert tag_for(contract['kind'], contract['lr'], contract['seed'], contract['temperature']) == tag
        if not compatible_runtime(r['environment'], environment()):
            raise RuntimeError('Recovery runtime differs. No record moved; inspect runtime first.')
        del r
        RESTORED_TRIALS.add(tag)
    elif allow_restart:
        RESTART_MISSING_TRIALS.add(tag)
    else:
        raise RuntimeError('No usable snapshot found. Restore from Bin first; restarting requires explicit allow_restart=True.')
    affected = [source]
    for name in ['M1_selected.json', 'temperature_ablation.json']:
        path = w / 'public' / name
        if path.exists() and any(x.get('tag') == tag for x in read(path)):
            affected.append(path)
    for path in affected:
        if path.exists():
            archive = w / 'public/recovery_incident/original_records' / (sha(path) + '_' + path.name)
            archive.parent.mkdir(parents=True, exist_ok=True)
            if not archive.exists():
                shutil.copyfile(path, archive)
            assert sha(archive) == sha(path), 'Archive bytes differ; active record preserved.'
            # Rename within Drive after verified copy; no permanent deletion.
            moved = archive.with_name('moved_' + archive.name)
            if moved.exists():
                raise RuntimeError('Archive destination already exists; inspect before continuing.')
            path.replace(moved)
    print('Authorized only:', tag, '| resume snapshot' if candidates else '| restart missing trial')
    print('Old metrics preserved under public/recovery_incident/original_records; not mixed into new results.')


def persist_recovery_metadata(path, record):
    import tempfile
    with tempfile.NamedTemporaryFile(mode='w', dir='/content/hiremind_upload_staging',
                                     suffix='.json', delete=False) as stream:
        json.dump(record, stream, indent=2)
        stream.flush()
        os.fsync(stream.fileno())
        local = Path(stream.name)
    if str(path).startswith('/content/drive/'):
        upload_verified(local, path, overwrite=True)
    else:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local, path)
    assert sha(path) == sha(local)
    local.unlink()

INCIDENT_M1_TAGS = {'M1_s43_lr5e-05_T2', 'M1_s42_lr0.0001_T2', 'M1_s42_lr0.0001_T4', 'M1_s42_lr5e-05_T2', 'M1_s44_lr5e-05_T2', 'M1_s42_lr5e-05_T4'}


## S07 · Models, evaluation and the original KD loss
Definitions only: running this cell does not start training.

In [ ]:
def tokens(work, part):
    t = torch.load(Path(work) / "private" / f"{part}_tokens.pt", weights_only=True)
    return TensorDataset(
        *(t[k] for k in ["input_ids", "attention_mask", "token_type_ids", "labels"]),
        torch.arange(len(t["labels"])),
    )

def batch_args(b, device, labels=True):
    ks = (
        ["input_ids", "attention_mask", "token_type_ids", "labels"]
        if labels
        else ["input_ids", "attention_mask", "token_type_ids"]
    )
    return {k: b[i].to(device) for i, k in enumerate(ks)}

def model(kind, seed_value, work):
    seed(seed_value)
    if kind == "teacher":
        f = Path(work) / "public/teacher_source.json"
        old = LEGACY / "public/teacher_source.json"
        if not f.exists():
            source = (
                read(old)
                if old.exists() and checked_legacy()
                else dict(model=TEACHER, revision=model_info(TEACHER).sha)
            )
            frozen_write(f, source)
        source = read(f)
        name, revision = source["model"], source["revision"]
    else:
        name, revision = STUDENT, STUDENT_REV
    return AutoModelForSequenceClassification.from_pretrained(
        name,
        revision=revision,
        num_labels=3,
        attn_implementation="eager",
        id2label=dict(enumerate(LABELS)),
        label2id={c: i for i, c in enumerate(LABELS)},
    )

def load_trial(work, info):
    if info.get('kind') == 'teacher':
        info = reviewed_trial(work, info, info.get('tag'))
    f = checkpoint_path(work, info)
    if sha(f) != info["weights_sha256"]:
        raise ValueError("Source weights changed.")
    m = model(info["kind"], info["seed"], work)
    m.load_state_dict(torch.load(f, map_location="cpu", weights_only=True), strict=True)
    return m.eval()

@torch.inference_mode()
def evaluate(m, ds, device, batch_size=16):
    m.eval()
    ys = []
    pred = []
    logits = []
    total = 0
    for b in DataLoader(ds, batch_size=batch_size, shuffle=False):
        out = m(**batch_args(b, device))
        assert torch.isfinite(out.logits).all()
        total += out.loss.item() * len(b[3])
        ys.extend(b[3].tolist())
        pred.extend(out.logits.argmax(-1).cpu().tolist())
        logits.append(out.logits.cpu())
    metrics = dict(
        loss=total / len(ys),
        accuracy=100 * accuracy_score(ys, pred),
        macro_f1=100 * f1_score(ys, pred, labels=[0, 1, 2], average="macro", zero_division=0),
        per_class=classification_report(
            ys, pred, labels=[0, 1, 2], target_names=LABELS, output_dict=True, zero_division=0
        ),
        confusion=confusion_matrix(ys, pred, labels=[0, 1, 2]).tolist(),
    )
    return metrics, torch.cat(logits), pred

def kd_loss(student_logits, labels, teacher_logits, temperature):
    ce = F.cross_entropy(student_logits, labels)
    kl = (
        F.kl_div(
            F.log_softmax(student_logits / temperature, dim=-1),
            F.softmax(teacher_logits / temperature, dim=-1),
            reduction="batchmean",
        )
        * temperature**2
    )
    return 0.5 * ce + 0.5 * kl, ce, kl

def select(runs):
    return max(runs, key=lambda r: (r["best"]["macro_f1"], -r["best"]["loss"]))

def quantize(m, variant):
    specs = {}
    if variant in ["linear", "both"]:
        specs[torch.nn.Linear] = default_dynamic_qconfig
    if variant in ["embeddings", "both"]:
        specs[torch.nn.Embedding] = float_qparams_weight_only_qconfig
    assert specs
    engines = torch.backends.quantized.supported_engines
    engine = "x86" if "x86" in engines else "fbgemm"
    if engine not in engines:
        raise RuntimeError("Quantization needs supported x86/fbgemm CPU backend.")
    torch.backends.quantized.engine = engine
    q = quantize_dynamic(m.cpu().eval(), qconfig_spec=specs, inplace=False)
    if variant in ["embeddings", "both"]:
        assert q.bert.embeddings.word_embeddings.weight().dtype == torch.quint8
    return q

def cost(m):
    c = m.config
    L = 256
    H = c.hidden_size
    I = c.intermediate_size
    return dict(
        parameters=sum(p.numel() for p in m.parameters()),
        macs=c.num_hidden_layers * (4 * L * H * H + 2 * L * L * H + 2 * L * H * I) + H * H + H * 3,
    )

def rss(field):
    for line in Path("/proc/self/status").read_text().splitlines():
        if line.startswith(field + ":"):
            return int(line.split()[1]) / 1024
    return None

def timing(fn, examples):
    for i in range(30):
        fn(examples[i % len(examples)])
    xs = []
    for i in range(200):
        t = time.perf_counter()
        fn(examples[i % len(examples)])
        xs.append((time.perf_counter() - t) * 1000)
    q1, q3 = np.percentile(xs, [25, 75])
    return dict(
        median_ms=float(np.median(xs)),
        q1_ms=float(q1),
        q3_ms=float(q3),
        iqr_ms=float(q3 - q1),
        p95_ms=float(np.percentile(xs, 95)),
        raw_ms=xs,
    )

def smoke(work):
    """Engineering test with random tensors. It must never enter the research results table."""
    w, p, v = paths(work)
    torch.set_num_threads(1)
    seed(7)
    cfg = BertConfig(
        vocab_size=30522,
        hidden_size=128,
        num_hidden_layers=2,
        num_attention_heads=2,
        intermediate_size=512,
        num_labels=3,
    )
    cfg._attn_implementation = "eager"
    m = BertForSequenceClassification(cfg).eval()
    x = dict(
        input_ids=torch.randint(0, 30522, (2, 256)),
        attention_mask=torch.ones(2, 256, dtype=torch.long),
        token_type_ids=torch.zeros(2, 256, dtype=torch.long),
    )
    s = torch.randn(4, 3, requires_grad=True)
    t = torch.randn(4, 3)
    y = torch.tensor([0, 1, 2, 0])
    loss, _, _ = kd_loss(s, y, t, 2.0)
    loss.backward()
    assert torch.isfinite(s.grad).all()
    exact = (
        F.kl_div(F.log_softmax(t / 2, dim=-1), F.softmax(t / 2, dim=-1), reduction="batchmean") * 4
    )
    assert abs(exact.item()) < 1e-6
    sizes = {}
    atomic_save(m.state_dict(), v / "random_fp32.pt")
    sizes["fp32"] = (v / "random_fp32.pt").stat().st_size
    for variant in ["linear", "embeddings", "both"]:
        q = quantize(m, variant)
        f = v / f"random_{variant}.pt"
        atomic_save(q.state_dict(), f)
        restored = quantize(m, variant)
        restored.load_state_dict(torch.load(f, weights_only=True))
        with torch.inference_mode():
            a = q(**x).logits
            b = restored(**x).logits
        assert torch.isfinite(b).all() and torch.allclose(a, b, atol=1e-6)
        sizes[variant] = f.stat().st_size
    result = dict(
        status="passed",
        scope="Random-initialized architecture and synthetic inputs ONLY; NOT trained model performance or course experimental results.",
        environment=environment(),
        checks=[
            "KD gradients finite",
            "KL identical-target zero",
            "all three quantization variants infer",
            "INT8 save/reload preserves logits",
            "embedding dtype quint8",
        ],
        serialized_bytes=sizes,
    )
    dump(p / "engineering_smoke_test.json", result)
    print(json.dumps(result, indent=2))

## S08 · Frozen data preparation and Item 2 checks
Definitions only: running this cell does not start training.

In [ ]:
def prepare(work):
    w, p, v = setup(work)
    rawfile = hf_hub_download(DATASET, "train.csv", repo_type="dataset", revision=DATA_REV)
    if sha(rawfile) != TRAIN_SHA:
        raise ValueError("Training CSV checksum mismatch.")
    raw = pd.read_csv(rawfile, keep_default_na=False)
    manifest = pd.read_csv(ROOT / "splits/splits.csv")
    check_splits(
        manifest
    )  # Original Item 2 audit function: ID-only checks, no test model evaluation.
    tok = AutoTokenizer.from_pretrained(STUDENT, revision=STUDENT_REV)
    frames = {}
    for part, n in [("train", 4993), ("validation", 1162)]:
        rows = manifest[manifest.split == part].copy()
        assert (rows.orig_split == "train").all()
        d = raw.iloc[rows.orig_index.to_numpy()].copy().reset_index(drop=True)
        assert len(d) == n and d.label.tolist() == rows.label.tolist()
        for k in ["row_uid", "cv_id", "job_id", "job_group", "label_id"]:
            d[k] = rows[k].to_numpy()
        assert d.resume_text.map(text_id).tolist() == d.cv_id.tolist()
        assert d.job_description_text.map(text_id).tolist() == d.job_id.tolist()
        frames[part] = d
        f = v / f"{part}_tokens.pt"
        meta = v / f"{part}_tokens.json"
        signature = dict(
            split_sha=SPLIT_SHA,
            train_sha=TRAIN_SHA,
            revision=STUDENT_REV,
            length=256,
            row_ids=d.row_uid.tolist(),
        )
        if f.exists() and meta.exists():
            rec = read(meta)
            if rec["signature"] != signature or sha(f) != rec["sha256"]:
                raise ValueError(f"Invalid {part} cache; preserved.")
            x = torch.load(f, weights_only=True)
            assert x["input_ids"].shape == (n, 256) and x["labels"].tolist() == d.label_id.tolist()
        else:
            out = {k: [] for k in ["input_ids", "attention_mask", "token_type_ids"]}
            for off in range(0, n, 64):
                b = d.iloc[off : off + 64]
                enc = tok(
                    b.resume_text.tolist(),
                    b.job_description_text.tolist(),
                    truncation="longest_first",
                    max_length=256,
                    padding="max_length",
                    return_tensors="pt",
                )
                for k in out:
                    out[k].append(enc[k])
            out = {k: torch.cat(x) for k, x in out.items()}
            out["labels"] = torch.tensor(d.label_id.tolist())
            if f.exists():
                old = torch.load(f, weights_only=True)
                assert all(
                    torch.equal(old[k], out[k]) for k in out
                ), "Uncommitted cache differs; retained."
            else:
                atomic_save(out, f)
            dump(meta, dict(signature=signature, sha256=sha(f)))
    tr, va = frames["train"], frames["validation"]
    selected = va.sample(n=20, random_state=42)
    frozen_write(
        v / "benchmark_inputs.json",
        selected[["row_uid", "resume_text", "job_description_text"]].to_dict("records"),
    )
    frozen_write(
        p / "data_used.json",
        dict(
            train_rows=4993,
            validation_rows=1162,
            test_rows=1759,
            split_sha256=SPLIT_SHA,
            training_row_ids=tr.row_uid.tolist(),
            validation_row_ids=va.row_uid.tolist(),
            benchmark_row_ids=selected.row_uid.tolist(),
            test_file_downloaded=False,
            test_evaluated=False,
            scope="Unseen jobs; possibly familiar CVs.",
        ),
    )
    sample = tr.groupby("label").sample(n=30, random_state=42).sample(frac=1, random_state=42)
    for person in ["Rawan", "Hams"]:
        f = v / f"label_audit_{person}.csv"
        if not f.exists():
            sample[["row_uid", "resume_text", "job_description_text"]].assign(
                human_label="", reason=""
            ).to_csv(f, index=False)
    if not (v / "label_audit_key.csv").exists():
        sample[["row_uid", "label"]].to_csv(v / "label_audit_key.csv", index=False)
    return frames

def baseline_references(frames, work):
    from sklearn.linear_model import LogisticRegression

    p = Path(work) / "public/item2_checks"
    p.mkdir(parents=True, exist_ok=True)
    if (p / "references.json").exists():
        return read(p / "references.json")
    tr, va = frames["train"], frames["validation"]
    major = int(tr.label_id.mode()[0])
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)
    x = vec.fit_transform(tr.resume_text + " [SEP] " + tr.job_description_text)
    y = vec.transform(va.resume_text + " [SEP] " + va.job_description_text)
    clf = LogisticRegression(max_iter=3000, random_state=42).fit(x, tr.label_id)

    def score(pred):
        return dict(
            macro_f1=100 * f1_score(va.label_id, pred, average="macro"),
            accuracy=100 * accuracy_score(va.label_id, pred),
        )

    result = dict(
        majority=score(np.full(len(va), major)),
        tfidf_logreg=score(clf.predict(y)),
        scope="New runtime checks; historical Item 2 evidence remains separate",
    )
    dump(p / "references.json", result)
    return result

def overfit_check(work):
    p = Path(work) / "public/item2_checks"
    f = p / "overfit.json"
    if f.exists():
        return read(f)
    seed(42)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    m = (
        AutoModelForSequenceClassification.from_pretrained(
            STUDENT,
            revision=STUDENT_REV,
            num_labels=3,
            attn_implementation="eager",
            hidden_dropout_prob=0.0,
            attention_probs_dropout_prob=0.0,
        )
        .to(device)
        .train()
    )
    ds = tokens(work, "train")
    idx = torch.randperm(len(ds), generator=torch.Generator().manual_seed(42))[:16]
    b = tuple(x[idx] for x in ds.tensors)
    args = batch_args(b, device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    losses = []
    for _ in range(300):
        out = m(**args)
        opt.zero_grad()
        out.loss.backward()
        opt.step()
        losses.append(out.loss.item())
    m.eval()
    with torch.inference_mode():
        acc = 100 * (m(**args).logits.argmax(-1) == args["labels"]).float().mean().item()
    result = dict(
        first_loss=losses[0],
        final_loss=losses[-1],
        final_accuracy=acc,
        losses=losses,
        steps=300,
        batch=16,
        passed=bool(losses[-1] < 0.05 and acc == 100),
        scope="Training sanity check only",
    )
    dump(f, result)
    del m, opt
    gc.collect()
    return result

def architecture_and_lengths(frames, work):
    p = Path(work) / "public/item2_checks"
    f = p / "architecture_lengths.json"
    if f.exists():
        return read(f)
    m = model("M0", 42, work)
    tok = AutoTokenizer.from_pretrained(STUDENT, revision=STUDENT_REV)
    from torch.utils.flop_counter import FlopCounterMode

    b = next(iter(DataLoader(tokens(work, "train"), batch_size=1)))
    with FlopCounterMode(display=False) as counter, torch.no_grad():
        m(**batch_args(b, "cpu", False))
    lengths = []
    for part in ["train", "validation"]:
        d = frames[part]
        total = []
        for off in range(0, len(d), 64):
            z = d.iloc[off : off + 64]
            enc = tok(
                z.resume_text.tolist(),
                z.job_description_text.tolist(),
                truncation=False,
                padding=False,
            )
            total.extend(map(len, enc["input_ids"]))
        lengths.extend(total)
    result = dict(
        **cost(m),
        measured_macs=counter.get_total_flops() / 2,
        embedding_parameters=sum(x.numel() for x in m.bert.embeddings.parameters()),
        token_pair_median=float(np.median(lengths)),
        token_pair_p90=float(np.percentile(lengths, 90)),
        pct_over_256=100 * float(np.mean(np.array(lengths) > 256)),
        lengths_scope="Frozen train + validation only; historical audit used its original scope",
    )
    dump(f, result)
    del m
    return result

def summarize_label_audit(work):
    from sklearn.metrics import cohen_kappa_score

    v = Path(work) / "private"
    a = pd.read_csv(v / "label_audit_Rawan.csv", keep_default_na=False)
    b = pd.read_csv(v / "label_audit_Hams.csv", keep_default_na=False)
    key = pd.read_csv(v / "label_audit_key.csv")
    if not (len(a) == len(b) == len(key) == 90):
        raise ValueError('Expected the same 90 audit rows in each independent sheet and key.')
    if not all(x.row_uid.is_unique for x in [a, b, key]):
        raise ValueError('Duplicate audit row IDs; review sheets before computing agreement.')
    assert a.row_uid.tolist() == b.row_uid.tolist() == key.row_uid.tolist()
    if not set(a.human_label) <= set(LABELS) or not set(b.human_label) <= set(LABELS):
        print(
            "PENDING: both independent label sheets need 90 valid labels. No agreement is invented."
        )
        return
    result = dict(
        n=90,
        kappa=float(cohen_kappa_score(a.human_label, b.human_label)),
        rawan_dataset_agreement=float(np.mean(a.human_label == key.label)),
        hams_dataset_agreement=float(np.mean(b.human_label == key.label)),
        note="Limited post-hoc training-label audit; disclose that original pre-training timing was not met if applicable.",
    )
    dump(Path(work) / "public/label_audit_summary.json", result)
    return result

## S09 · Visible resumable training loop
Definitions only: running this cell does not start training.
Full recovery files contain optimizer/RNG state and must only be loaded from your own trusted run folder.

In [ ]:
def train_trial(work, kind, lr, s, temperature=None):
    w, p, v = setup(work)
    cached = completed_trial(work, kind, lr, s, temperature)
    if cached:
        print("REUSE completed:", cached["tag"])
        return cached
    if not STORAGE_READY:
        raise RuntimeError('Run C02 to verify cloud storage before any new training.')
    if tag_for(kind, lr, s, temperature) in INCIDENT_M1_TAGS:
        tag_check = tag_for(kind, lr, s, temperature)
        if tag_check not in RESTORED_TRIALS and tag_check not in RESTART_MISSING_TRIALS:
            raise RuntimeError('Historical M1 trial is missing valid weights. Review C01 and explicitly authorize only this trial in C03: ' + tag_check)
    if (p / "final_freeze.json").exists():
        raise RuntimeError("Protocol frozen for final test: no further training in this cohort.")
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime to run incomplete training trials.")
    env = training_environment_guard(work)
    contract = trial_contract(kind, lr, s, temperature)
    tag = tag_for(kind, lr, s, temperature)
    tr = tokens(work, "train")
    va = tokens(work, "validation")
    m = model(kind, s, work).cuda()
    g = torch.Generator().manual_seed(s)
    loader = DataLoader(tr, batch_size=16, shuffle=True, generator=g, num_workers=0)
    total = 3 * len(loader)
    assert total == 939
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    teacher_logits = None
    if kind == "M1":
        teacher_provenance = verified_teacher_targets(work, pin=True)
        cache = torch.load(v / "teacher_train_logits.pt", weights_only=True)
        assert cache["training_row_ids"] == read(p / "data_used.json")["training_row_ids"]
        assert (
            cache["teacher_weights_sha256"] == read(p / "teacher_selected.json")["weights_sha256"]
        )
        teacher_logits = cache["logits"]
        assert teacher_logits.shape == (len(tr), 3)
    current_elapsed = 0.0
    step = 0
    history = []
    running = []
    best = {"macro_f1": -1.0}
    best_state = None
    start_epoch = 0
    skip = 0
    elapsed = 0.0
    peak = 0.0
    recovery = None
    valid = recovery_candidates(work, tag)
    for _, f, _ in valid:
        try:
            # Only load our own checksum-validated recovery files: optimizer/RNG data require pickle.
            r = torch.load(f, map_location="cpu", weights_only=False)
            assert (
                r["contract"] == contract and compatible_runtime(r["environment"], env) and r["recovery_version"] == 2
            )
            if kind == 'M1' and r.get('teacher_provenance') is not None:
                assert r['teacher_provenance'] == teacher_provenance
            m.load_state_dict(r["model"])
            opt.load_state_dict(r["optimizer"])
            sch.load_state_dict(r["scheduler"])
            recovery = r
            break
        except Exception as ex:
            print("Recovery not usable; preserved:", f.name, ex)
    if valid and recovery is None:
        raise RuntimeError(
            "Recovery exists but failed validation; inspect it rather than silently restarting."
        )
    if recovery:
        r = recovery
        step = r["step"]
        history = r["history"]
        running = r["running"]
        best = r["best"]
        best_state = r["best_state"]
        start_epoch = r["epoch"]
        skip = r["batches_done"]
        elapsed = r["elapsed"]
        current_elapsed = elapsed
        peak = r["peak"]
        g.set_state(r["epoch_generator"])
        print("RESUME", tag, "from update", step)
    elif (LEGACY / "private/checkpoints" / f"{tag}.pt").exists():
        print(
            "RESTART ONLY incomplete legacy trial:",
            tag,
            "(old best weights retained; no optimizer/RNG state).",
        )
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    last_saved_step = step
    micro = 4 if kind == "teacher" else 16
    for epoch in range(start_epoch, 3):
        epoch_generator = g.get_state().clone()
        iterator = iter(loader)
        # Recreate the SAME permutation from the epoch-start generator; consume completed batches.
        for _ in range(skip):
            next(iterator)
        if recovery:
            restore_rng(recovery["rng"])
            recovery = None
        m.train()
        for batch_no, b in enumerate(iterator, start=skip):
            opt.zero_grad(set_to_none=True)
            batch_loss = 0.0
            for off in range(0, len(b[3]), micro):
                bb = tuple(t[off : off + micro] for t in b)
                args = batch_args(bb, "cuda")
                out = m(**args)
                loss = (
                    out.loss
                    if teacher_logits is None
                    else kd_loss(
                        out.logits, args["labels"], teacher_logits[bb[4]].cuda(), temperature
                    )[0]
                )
                fraction = len(bb[3]) / len(b[3])
                (loss * fraction).backward()
                batch_loss += loss.item() * fraction
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sch.step()
            step += 1
            running.append(batch_loss)
            if step % 100 == 0 or step == total:
                met, _, _ = evaluate(m, va, "cuda")
                history.append(
                    dict(
                        step=step,
                        epoch=epoch + 1,
                        train_objective=float(np.mean(running)),
                        validation_ce=met["loss"],
                        validation_macro_f1=met["macro_f1"],
                        validation_accuracy=met["accuracy"],
                    )
                )
                running = []
                if met["macro_f1"] > best["macro_f1"]:
                    best = {**met, "step": step}
                    best_state = {k: x.detach().cpu().clone() for k, x in m.state_dict().items()}
                m.train()
                torch.cuda.synchronize()
                current_elapsed = elapsed + time.perf_counter() - started
                peak = max(peak, torch.cuda.max_memory_allocated() / 2**20)
                state = dict(
                    recovery_version=2,
                    teacher_provenance=teacher_provenance if kind == "M1" else None,
                    contract=contract,
                    environment=env,
                    model={k: x.detach().cpu().clone() for k, x in m.state_dict().items()},
                    optimizer=opt.state_dict(),
                    scheduler=sch.state_dict(),
                    rng=capture_rng(),
                    epoch=epoch,
                    batches_done=batch_no + 1,
                    epoch_generator=epoch_generator,
                    step=step,
                    history=history,
                    running=running,
                    best=best,
                    best_state=best_state,
                    elapsed=current_elapsed,
                    peak=peak,
                )
                save_recovery(work, tag, state)
                last_saved_step = step
                dump(p / "curves" / f"{tag}_step{step}.json", history)
                print(
                    tag,
                    "update",
                    step,
                    "validation macro-F1",
                    round(met["macro_f1"], 3),
                    "recovery saved",
                    flush=True,
                )
        skip = 0
    assert best_state is not None and step == total
    weights = v / "checkpoints" / f"{tag}_{uuid.uuid4().hex[:8]}.pt"
    atomic_save(best_state, weights)
    info = dict(
        tag=tag,
        kind=kind,
        lr=lr,
        seed=s,
        temperature=temperature,
        steps=step,
        best=best,
        seconds=current_elapsed,
        timing_scope="training/validation elapsed; includes earlier recovery I/O; interrupted unsaved work excluded",
        peak_gpu_allocated_MiB=peak,
        weights=str(weights.resolve()),
        weights_sha256=sha(weights),
        history=history,
        effective_batch=16,
        micro_batch=micro,
        validation_batch=16,
        precision="float32",
        contract=contract,
        actual_environment=env,
        teacher_provenance=teacher_provenance if kind == "M1" else None,
    )
    frozen_write(p / "trials" / f"{tag}.json", info)
    del m, opt, sch, best_state
    gc.collect()
    torch.cuda.empty_cache()
    return validate_trial(work, info)


## S10 · M0, teacher, M1 and temperature experiment orchestration
Definitions only: running this cell does not start training.

In [ ]:
def train_m0(work):
    _, p, _ = setup(work)
    runs = [train_trial(work, "M0", lr, 42) for lr in CONFIG["m0_lrs"]]
    best = select(runs)
    selected = [best] + [train_trial(work, "M0", best["lr"], s) for s in CONFIG["seeds"][1:]]
    frozen_write(p / "M0_selected.json", selected)

def train_teacher(work):
    w, p, v = setup(work)
    if (p / 'teacher_selected.json').exists():
        binding = verified_teacher_targets(work, pin=True)
        print('REUSE ORIGINAL TEACHER TARGETS:', binding['teacher_tag'])
        info = reviewed_trial(w, read(p / 'teacher_selected.json'))
        print('REUSE VERIFIED RECOVERED TEACHER:', info['weights']) if info.get('recovery_verified') else print('Original Teacher weights verified.')
        print('Original cached training targets remain unchanged. No Teacher training performed.')
        return
    runs = [train_trial(work, "teacher", lr, 42) for lr in CONFIG["teacher_lrs"]]
    best = select(runs)
    frozen_write(p / "teacher_selected.json", best)
    cf = v / "teacher_train_logits.pt"
    ids = read(p / "data_used.json")["training_row_ids"]
    for f in [cf, LEGACY / "private/teacher_train_logits.pt"]:
        if not f.exists():
            continue
        try:
            x = torch.load(f, map_location="cpu", weights_only=True)
            assert (
                x["training_row_ids"] == ids
                and x["teacher_weights_sha256"] == best["weights_sha256"]
            )
            assert x["logits"].shape == (4993, 3) and torch.isfinite(x["logits"]).all()
            if f != cf:
                atomic_save(x, cf)
            print("REUSE verified teacher targets")
            return
        except Exception as ex:
            if f == cf:
                raise RuntimeError(
                    "Existing target cache invalid; preserved for inspection."
                ) from ex
            print("Legacy cache not reused:", ex)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    m = load_trial(work, best).to(device)
    _, logits, _ = evaluate(m, tokens(work, "train"), device)
    atomic_save(
        dict(logits=logits, training_row_ids=ids, teacher_weights_sha256=best["weights_sha256"]), cf
    )
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def train_m1(work):
    _, p, _ = setup(work)
    verified_teacher_targets(work, pin=True)
    runs = [
        train_trial(work, "M1", lr, 42, t)
        for lr in CONFIG["m1_lrs"]
        for t in CONFIG["temperatures"]
    ]
    best = select(runs)
    selected = [best] + [
        train_trial(work, "M1", best["lr"], s, best["temperature"]) for s in CONFIG["seeds"][1:]
    ]
    frozen_write(p / "M1_selected.json", selected)

def temperature_ablation(work):
    _, p, _ = setup(work)
    verified_teacher_targets(work, pin=True)
    best = read(p / "M1_selected.json")[0]
    runs = [train_trial(work, "M1", best["lr"], s, 1.0) for s in CONFIG["seeds"]]
    frozen_write(p / "temperature_ablation.json", runs)


## S11 · Quantization and clean-process CPU measurement
Definitions only: running this cell does not start training.

In [ ]:
def quantize_runs(work):
    w, p, v = setup(work)
    torch.set_num_threads(1)
    rows = []
    val = tokens(work, "validation")
    for info in read(p / "M1_selected.json"):
        fp = load_trial(work, info)
        _, _, base = evaluate(fp, val, "cpu")
        counts = cost(fp)
        for variant in ["linear", "embeddings", "both"]:
            tag = f"Q_{variant}_s{info['seed']}"
            f = p / "quantization" / f"{tag}.json"
            if f.exists():
                row = read(f)
                wf = checkpoint_path(work, row)
                assert (
                    row["source_sha256"] == info["weights_sha256"]
                    and sha(wf) == row["weights_sha256"]
                )
                assert row["validation_ids"] == read(p / "data_used.json")["validation_row_ids"]
                q = quantize(fp, variant)
                q.load_state_dict(torch.load(wf, map_location="cpu", weights_only=True))
                del q
                rows.append(row)
                print("REUSE quantized result:", tag)
                continue
            q = quantize(fp, variant)
            qf = v / "checkpoints" / f"{tag}_{uuid.uuid4().hex[:8]}.pt"
            atomic_save(q.state_dict(), qf)
            loaded = quantize(fp, variant)
            loaded.load_state_dict(torch.load(qf, map_location="cpu", weights_only=True))
            b = next(iter(DataLoader(val, batch_size=4)))
            with torch.inference_mode():
                assert torch.allclose(
                    q(**batch_args(b, "cpu", False)).logits,
                    loaded(**batch_args(b, "cpu", False)).logits,
                    atol=1e-6,
                    rtol=1e-5,
                )
            met, logits, pred = evaluate(loaded, val, "cpu")
            row = dict(
                tag=tag,
                variant=variant,
                seed=info["seed"],
                source=info["tag"],
                source_sha256=info["weights_sha256"],
                weights=str(qf.resolve()),
                weights_sha256=sha(qf),
                metrics=met,
                serialized_bytes=qf.stat().st_size,
                **counts,
                prediction_agreement=float(np.mean(np.array(pred) == np.array(base))),
                reload_verified=True,
                prediction_agreement_at_least_95=bool(
                    np.mean(np.array(pred) == np.array(base)) >= 0.95
                ),
                validation_ids=read(p / "data_used.json")["validation_row_ids"],
            )
            frozen_write(f, row)
            rows.append(row)
            del q, loaded
            gc.collect()
        del fp
    frozen_write(p / "quantization_results.json", rows)

def run_benchmarks(work):
    w, p, v = setup(work)
    records = read(p / "M0_selected.json") + read(p / "M1_selected.json")
    qr = read(p / "quantization_results.json")
    spec = dict(
        environment=environment(),
        input_hash=sha(v / "benchmark_inputs.json"),
        threads=1,
        length=256,
        warmup=30,
        runs=200,
        sources=[x["weights_sha256"] for x in records + qr],
    )
    key = json_hash(spec)
    folder = p / "benchmark_sessions" / key
    folder.mkdir(parents=True, exist_ok=True)
    support = [
        "read",
        "dump",
        "sha",
        "seed",
        "environment",
        "model",
        "checkpoint_path",
        "load_trial",
        "quantize",
        "cost",
        "rss",
        "timing",
    ]
    # IPython preserves the readable definition cells in linecache; fail visibly if unavailable.
    source = "import os,json,time,gc,hashlib,random,platform\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch,transformers\nfrom transformers import AutoModelForSequenceClassification,AutoTokenizer\nfrom huggingface_hub import model_info\nfrom torch.ao.quantization import quantize_dynamic,default_dynamic_qconfig,float_qparams_weight_only_qconfig\n"
    source += f"LABELS={LABELS!r}\nSTUDENT={STUDENT!r}\nSTUDENT_REV={STUDENT_REV!r}\nTEACHER={TEACHER!r}\n"
    for name in support:
        source += "\n" + visible_source(globals()[name]) + "\n"
    source += "\n" + visible_source(cpu_worker) + "\n"
    for s in [42, 43, 44]:
        for kind, variant in [
            ("M0", "fp32"),
            ("M1", "fp32"),
            ("M1", "linear"),
            ("M1", "embeddings"),
            ("M1", "both"),
        ]:
            out = folder / f"{kind}_s{s}_{variant}.json"
            if out.exists():
                r = read(out)
                assert r["benchmark_spec"] == spec
                assert (
                    len(r["model_only"]["raw_ms"]) == 200
                    and len(r["tokenization_inclusive"]["raw_ms"]) == 200
                )
                continue
            cmd = (
                source
                + f"\ncpu_worker({str(w)!r},{kind!r},{s!r},{variant!r},{str(out)!r},{spec!r})\n"
            )
            subprocess.run([sys.executable, "-c", cmd], check=True)
    dump(p / "active_benchmark.json", dict(folder=str(folder.resolve()), spec=spec))
    print("Complete comparable CPU benchmark session:", folder)

def cpu_worker(work, kind, s, variant, out, spec):
    torch.set_num_threads(1)
    w = Path(work)
    p = w / "public"
    v = w / "private"
    imports = rss("VmRSS")
    info = next(x for x in read(p / f"{kind}_selected.json") if x["seed"] == s)
    fp = load_trial(work, info)
    counts = cost(fp)
    if variant == "fp32":
        m = fp
        file = Path(info["weights"])
    else:
        row = next(
            x
            for x in read(p / "quantization_results.json")
            if x["seed"] == s and x["variant"] == variant
        )
        file = Path(row["weights"])
        assert sha(file) == row["weights_sha256"]
        m = quantize(fp, variant)
        m.load_state_dict(torch.load(file, map_location="cpu", weights_only=True))
        del fp
        gc.collect()
    tok = AutoTokenizer.from_pretrained(STUDENT, revision=STUDENT_REV)
    examples = read(v / "benchmark_inputs.json")

    def encode(x):
        return tok(
            x["resume_text"],
            x["job_description_text"],
            truncation="longest_first",
            max_length=256,
            padding="max_length",
            return_tensors="pt",
        )

    enc = [encode(x) for x in examples]
    m.eval()
    with torch.inference_mode():
        only = timing(lambda x: m(**x), enc)
        inclusive = timing(lambda x: m(**encode(x)), examples)
    dump(
        out,
        dict(
            kind=kind,
            variant=variant,
            seed=s,
            benchmark_spec=spec,
            model_only=only,
            tokenization_inclusive=inclusive,
            process_peak_MiB=rss("VmHWM"),
            imports_rss_MiB=imports,
            post_inference_rss_MiB=rss("VmRSS"),
            memory_scope="Fresh process peak including load, tokenization, conversion/reload and inference; not inference-only peak.",
            serialized_bytes=file.stat().st_size,
            **counts,
        ),
    )

## S12 · Tables, figures and error analysis
Definitions only: running this cell does not start training.

In [ ]:
def comparison_tables(work, partition="validation"):
    p = Path(work) / "public"
    out = p / "reports" / partition
    out.mkdir(parents=True, exist_ok=True)
    bench = Path(read(p / "active_benchmark.json")["folder"])
    rows = []
    for kind in ["M0", "M1"]:
        for x in read(p / f"{kind}_selected.json"):
            met = (
                x["best"]
                if partition == "validation"
                else read(p / "final_test" / f"{kind}_s{x['seed']}.json")["metrics"]
            )
            rows.append(
                dict(
                    model=kind,
                    kind=kind,
                    variant="fp32",
                    seed=x["seed"],
                    accuracy=met["accuracy"],
                    macro_f1=met["macro_f1"],
                    good_fit_recall=100 * met["per_class"]["Good Fit"]["recall"],
                )
            )
    for x in read(p / "quantization_results.json"):
        if partition == "test" and x["variant"] != "both":
            continue
        name = {"both": "M2", "linear": "Linear only", "embeddings": "Embeddings only"}[
            x["variant"]
        ]
        met = (
            x["metrics"]
            if partition == "validation"
            else read(p / "final_test" / f"M2_s{x['seed']}.json")["metrics"]
        )
        rows.append(
            dict(
                model=name,
                kind="M1",
                variant=x["variant"],
                seed=x["seed"],
                accuracy=met["accuracy"],
                macro_f1=met["macro_f1"],
                good_fit_recall=100 * met["per_class"]["Good Fit"]["recall"],
            )
        )
    for r in rows:
        b = read(bench / f"{r['kind']}_s{r['seed']}_{r['variant']}.json")
        r.update(
            params_M=b["parameters"] / 1e6,
            size_MB=b["serialized_bytes"] / 1e6,
            size_MiB=b["serialized_bytes"] / 2**20,
            macs_G=b["macs"] / 1e9,
            latency_ms=b["model_only"]["median_ms"],
            q1_ms=b["model_only"]["q1_ms"],
            q3_ms=b["model_only"]["q3_ms"],
            iqr_ms=b["model_only"]["iqr_ms"],
            inclusive_ms=b["tokenization_inclusive"]["median_ms"],
            peak_process_MiB=b["process_peak_MiB"],
        )
    df = pd.DataFrame(rows)
    assert not df.isna().any().any()
    df.to_csv(out / "per_seed.csv", index=False)
    numeric = [
        "accuracy",
        "macro_f1",
        "good_fit_recall",
        "params_M",
        "size_MB",
        "macs_G",
        "latency_ms",
        "iqr_ms",
        "peak_process_MiB",
    ]
    df.groupby("model")[numeric].agg(["mean", "std"]).to_csv(out / "summary_mean_sample_SD.csv")
    paired = []
    for s in [42, 43, 44]:
        d = df[df.seed == s].set_index("model")
        a, b, c = [d.loc[k] for k in ["M0", "M1", "M2"]]
        paired.append(
            dict(
                seed=s,
                M1_minus_M0=b.macro_f1 - a.macro_f1,
                M2_minus_M1=c.macro_f1 - b.macro_f1,
                M2_minus_M0=c.macro_f1 - a.macro_f1,
                size_ratio=c.size_MB / b.size_MB,
                latency_ratio=c.latency_ms / b.latency_ms,
            )
        )
    pd.DataFrame(paired).to_csv(out / "paired_changes.csv", index=False)
    import matplotlib.pyplot as plt

    plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for name in ["M0", "M1", "M2"]:
        d = df[df.model == name]
        ax.errorbar(
            d.latency_ms,
            d.accuracy,
            xerr=[d.latency_ms - d.q1_ms, d.q3_ms - d.latency_ms],
            fmt="o",
            capsize=3,
            label=name,
        )
        for _, r in d.iterrows():
            ax.annotate(
                str(r.seed),
                (r.latency_ms, r.accuracy),
                xytext=(3, 3),
                textcoords="offset points",
                fontsize=8,
            )
    ax.set(xlabel="CPU latency (ms): median and Q1–Q3", ylabel=f"{partition.title()} accuracy (%)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "accuracy_latency.png", dpi=200)
    plt.show()
    if partition == "validation":
        order = ["M1", "Linear only", "Embeddings only", "M2"]
        g = df.groupby("model").macro_f1
        fig, ax = plt.subplots(figsize=(6.5, 4))
        ax.bar(order, g.mean().reindex(order), yerr=g.std().reindex(order), capsize=4)
        for s in [42, 43, 44]:
            ax.plot(
                order,
                [df[(df.model == k) & (df.seed == s)].macro_f1.iloc[0] for k in order],
                ".-",
                alpha=0.65,
                label=f"Seed {s}",
            )
        ax.set(ylabel="Validation macro-F1 (%)", ylim=(0, 100))
        ax.legend()
        fig.tight_layout()
        fig.savefig(out / "quantization_ablation.png", dpi=200)
        plt.show()
        t = read(p / "temperature_ablation.json")
        pd.DataFrame(
            [
                dict(
                    seed=x["seed"],
                    temperature=1,
                    macro_f1=x["best"]["macro_f1"],
                    accuracy=x["best"]["accuracy"],
                )
                for x in t
            ]
        ).to_csv(out / "temperature_ablation.csv", index=False)
    print(df.to_string(index=False))
    return df

def selected_models(work):
    p = Path(work) / "public"
    result = []
    for k in ["M0", "M1"]:
        for x in read(p / f"{k}_selected.json"):
            result.append((k, x))
    for x in read(p / "M1_selected.json"):
        result.append(("M2", x))
    return result

def error_analysis(work, frames, partition="validation"):
    import matplotlib.pyplot as plt

    w = Path(work)
    p = w / "public"
    out = p / "error_analysis" / partition
    out.mkdir(parents=True, exist_ok=True)
    frame = frames[partition]
    ds = tokens(work, partition)
    for name, info in selected_models(work):
        tag = f"{name}_s{info['seed']}"
        f = out / f"{tag}.json"
        if f.exists():
            previous = read(f)
            assert previous["source_sha256"] == info["weights_sha256"]
            if (out / f"{tag}_confusion.png").exists() and (
                out / f"{tag}_predictions.csv"
            ).exists():
                continue
        m = None
        if partition == "test":
            saved = read(p / "final_test" / f"{tag}.json")
            assert saved["source_sha256"] == info["weights_sha256"]
            assert saved["row_ids"] == frame.row_uid.tolist()
            met, pred = saved["metrics"], saved["predictions"]
            confidence = np.asarray(saved["confidence"])
        else:
            m = load_trial(work, info)
            if name == "M2":
                row = next(
                    x
                    for x in read(p / "quantization_results.json")
                    if x["seed"] == info["seed"] and x["variant"] == "both"
                )
                assert sha(row["weights"]) == row["weights_sha256"]
                m = quantize(m, "both")
                m.load_state_dict(torch.load(row["weights"], map_location="cpu", weights_only=True))
            met, logits, pred = evaluate(m, ds, "cpu")
            prob = torch.softmax(logits, dim=-1).numpy()
            confidence = prob.max(1)
        public = pd.DataFrame(
            dict(row_uid=frame.row_uid, true=frame.label_id, predicted=pred, confidence=confidence)
        )
        public.to_csv(out / f"{tag}_predictions.csv", index=False)
        dump(f, dict(metrics=met, partition=partition, source_sha256=info["weights_sha256"]))
        cm = np.array(met["confusion"])
        fig, ax = plt.subplots(figsize=(4.8, 4))
        ax.imshow(cm, cmap="Blues")
        for i in range(3):
            for j in range(3):
                ax.text(
                    j,
                    i,
                    str(cm[i, j]),
                    ha="center",
                    va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                )
        ax.set_xticks(range(3), LABELS, rotation=20)
        ax.set_yticks(range(3), LABELS)
        ax.set(xlabel="Predicted label", ylabel="Dataset label", title=f"{tag}: {partition}")
        fig.tight_layout()
        fig.savefig(out / f"{tag}_confusion.png", dpi=200)
        plt.close(fig)
        errors = (
            public[public.true != public.predicted]
            .sort_values("confidence", ascending=False)
            .head(15)
        )
        private = errors.merge(
            frame[["row_uid", "resume_text", "job_description_text"]], on="row_uid"
        ).assign(observed_issue="", possible_explanation="", evidence="")
        d = w / "private/error_review" / partition
        d.mkdir(parents=True, exist_ok=True)
        review_file = d / f"{tag}_manual_review.csv"
        if not review_file.exists():
            private.to_csv(review_file, index=False)
        del m
        gc.collect()
    print(
        "Confusions/class reports saved. Private error-review sheets require genuine human analysis; confidence is not calibrated hiring probability."
    )

def training_diagnostics(work):
    """Reuse saved histories; no training is performed."""
    import matplotlib.pyplot as plt

    p = Path(work) / "public"
    out = p / "reports/validation"
    out.mkdir(parents=True, exist_ok=True)
    search = []
    for lr in CONFIG["m0_lrs"]:
        r = completed_trial(work, "M0", lr, 42)
        if r is None:
            raise RuntimeError("M0 search is incomplete.")
        search.append(
            dict(
                lr=lr,
                macro_f1=r["best"]["macro_f1"],
                accuracy=r["best"]["accuracy"],
                best_step=r["best"]["step"],
            )
        )
    pd.DataFrame(search).to_csv(out / "M0_learning_rate_search.csv", index=False)
    for kind in ["M0", "M1"]:
        selected = read(p / f"{kind}_selected.json")
        for run in selected:
            h = pd.DataFrame(run["history"])
            fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
            axes[0].plot(h.step, h.train_objective, label="Training objective")
            axes[0].plot(h.step, h.validation_ce, label="Validation CE")
            axes[0].set(xlabel="Update", ylabel="Loss")
            axes[0].legend()
            axes[1].plot(h.step, h.validation_macro_f1)
            axes[1].set(xlabel="Update", ylabel="Validation macro-F1 (%)")
            fig.suptitle(
                f"{kind}, seed {run['seed']}: training objective differs from validation CE for M1"
            )
            fig.tight_layout()
            fig.savefig(out / f"{kind}_seed{run['seed']}_curves.png", dpi=200)
            plt.close(fig)
    teacher = read(p / "teacher_selected.json")
    m0 = read(p / "M0_selected.json")
    dump(
        out / "teacher_context.json",
        dict(
            teacher_validation_macro_f1=teacher["best"]["macro_f1"],
            M0_seed42_macro_f1=m0[0]["best"]["macro_f1"],
            note="One selected teacher, not a three-teacher seed study; inspect its strength without replacing the approved method.",
        ),
    )

## S13 · Item 4 final-test gate and final hypothesis checks
Definitions only: running this cell does not start training.

In [ ]:
def freeze_final(work, confirmation):
    if confirmation != "FREEZE AFTER ITEM 3 REVIEW":
        raise ValueError("Review Item 3 and explicitly set the freeze phrase first.")
    p = Path(work) / "public"
    selection = {}
    for name in [
        "M0_selected",
        "M1_selected",
        "quantization_results",
        "temperature_ablation",
        "active_benchmark",
    ]:
        f = p / f"{name}.json"
        selection[name] = sha(f)
    for name, info in selected_models(work):
        validate_trial(work, info)
    if not (p / "reports/validation/per_seed.csv").exists():
        raise ValueError("Finish Item 3 tables first.")
    if len(list((p / "error_analysis/validation").glob("*_confusion.png"))) != 9:
        raise ValueError("Complete validation error analysis before freeze.")
    record = dict(
        protocol=protocol(),
        selection_hashes=selection,
        test_sha256=TEST_SHA,
        seeds=[42, 43, 44],
        deployment_seed=42,
        rule="No selection or tuning using test. H1/H2 assessed on paired three-seed means. Ablations reported on validation.",
    )
    frozen_write(p / "final_freeze.json", record)
    print("Final choices frozen; subsequent test results cannot be used for tuning this cohort.")

def final_test(work):
    w, p, v = setup(work)
    freeze = read(p / "final_freeze.json")
    assert freeze["protocol"] == protocol()
    for name, h in freeze["selection_hashes"].items():
        assert sha(p / f"{name}.json") == h, "Selection changed after freeze."
    f = hf_hub_download(DATASET, "test.csv", repo_type="dataset", revision=DATA_REV)
    assert sha(f) == TEST_SHA
    raw = pd.read_csv(f, keep_default_na=False)
    ids = pd.read_csv(ROOT / "splits/splits.csv")
    ids = ids[ids.split == "test"]
    assert (ids.orig_split == "test").all()
    d = raw.iloc[ids.orig_index.to_numpy()].copy().reset_index(drop=True)
    assert len(d) == 1759 and d.label.tolist() == ids.label.tolist()
    for k in ["row_uid", "cv_id", "job_id", "job_group", "label_id"]:
        d[k] = ids[k].to_numpy()
    assert (
        d.resume_text.map(text_id).tolist() == d.cv_id.tolist()
        and d.job_description_text.map(text_id).tolist() == d.job_id.tolist()
    )
    tok = AutoTokenizer.from_pretrained(STUDENT, revision=STUDENT_REV)
    out = {k: [] for k in ["input_ids", "attention_mask", "token_type_ids"]}
    for off in range(0, len(d), 64):
        b = d.iloc[off : off + 64]
        x = tok(
            b.resume_text.tolist(),
            b.job_description_text.tolist(),
            truncation="longest_first",
            max_length=256,
            padding="max_length",
            return_tensors="pt",
        )
        for k in out:
            out[k].append(x[k])
    out = {k: torch.cat(x) for k, x in out.items()}
    out["labels"] = torch.tensor(d.label_id.tolist())
    tf = v / "test_tokens.pt"
    if not tf.exists():
        atomic_save(out, tf)
    else:
        assert all(torch.equal(torch.load(tf, weights_only=True)[k], out[k]) for k in out)
    for name, info in selected_models(work):
        dst = p / "final_test" / f"{name}_s{info['seed']}.json"
        if dst.exists():
            r = read(dst)
            assert r["freeze_hash"] == json_hash(freeze)
            assert r["source_sha256"] == info["weights_sha256"]
            assert r["row_ids"] == d.row_uid.tolist()
            assert len(r["predictions"]) == len(d)
            continue
        m = load_trial(work, info)
        if name == "M2":
            row = next(
                x
                for x in read(p / "quantization_results.json")
                if x["seed"] == info["seed"] and x["variant"] == "both"
            )
            assert sha(row["weights"]) == row["weights_sha256"]
            m = quantize(m, "both")
            m.load_state_dict(torch.load(row["weights"], map_location="cpu", weights_only=True))
        met, logits, pred = evaluate(m, tokens(work, "test"), "cpu")
        frozen_write(
            dst,
            dict(
                metrics=met,
                freeze_hash=json_hash(freeze),
                source_sha256=info["weights_sha256"],
                row_ids=d.row_uid.tolist(),
                predictions=pred,
                confidence=torch.softmax(logits, dim=-1).max(dim=-1).values.tolist(),
            ),
        )
        del m
        gc.collect()
    return {"test": d}

def final_hypotheses(work):
    p = Path(work) / "public"
    df = pd.read_csv(p / "reports/test/per_seed.csv")
    mean = df.groupby("model").macro_f1.mean()
    gain = mean["M1"] - mean["M0"]
    loss = mean["M1"] - mean["M2"]
    kept = mean["M2"] - mean["M0"]
    ratios = []
    for s in [42, 43, 44]:
        d = df[df.seed == s].set_index("model")
        ratios.append(float(d.loc["M2", "size_MB"] / d.loc["M1", "size_MB"]))
    result = dict(
        H1_gain_at_least_one_point=bool(gain >= 1),
        gain_points=float(gain),
        H2_storage_all_seeds=bool(max(ratios) <= 0.4),
        size_ratios=ratios,
        H2_quality_loss_at_most_one_point=bool(loss <= 1),
        quality_loss_points=float(loss),
        gain_retention_confirmatory=bool(gain >= 1),
        retained_gain_points=float(kept),
        retention_threshold_met=bool(kept >= 0.5 * gain) if gain >= 1 else None,
        note="Descriptive three-seed evidence, not a statistical-significance claim. When H1 fails, retention is not confirmatory.",
    )
    frozen_write(p / "final_test/hypotheses.json", result)
    return result

## S14 · Portable Senior CPU component and evidence export
Definitions only: running this cell does not start training.

In [ ]:
class CPUFitPredictor:
    """Portable text-pair classifier. No training, ranking, assessments or hiring decision."""

    def __init__(self, directory):
        from pathlib import Path
        import json, hashlib
        import torch, transformers
        from transformers import BertConfig, BertForSequenceClassification, AutoTokenizer
        from torch.ao.quantization import (
            quantize_dynamic,
            default_dynamic_qconfig,
            float_qparams_weight_only_qconfig,
        )

        self.torch = torch
        self.directory = Path(directory)
        self.meta = json.loads((self.directory / "model_manifest.json").read_text())
        self.labels = self.meta["labels"]
        if self.labels != ["No Fit", "Potential Fit", "Good Fit"] or self.meta["max_length"] != 256:
            raise ValueError("Unexpected input/label contract.")
        for name, expected in self.meta["files"].items():
            f = self.directory / name
            h = hashlib.sha256()
            with f.open("rb") as stream:
                for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                    h.update(block)
            if h.hexdigest() != expected:
                raise ValueError(f"Artifact checksum mismatch: {name}")
        expected = self.meta["runtime"]
        if (
            torch.__version__.split("+")[0] != expected["torch"].split("+")[0]
            or transformers.__version__ != expected["transformers"]
        ):
            raise RuntimeError(
                "Use the exported PyTorch base version and Transformers version; test compatibility before upgrading."
            )
        torch.set_num_threads(1)
        config = BertConfig.from_pretrained(self.directory, local_files_only=True)
        config._attn_implementation = "eager"
        self.model = BertForSequenceClassification(config).cpu().eval()
        if self.meta["variant"] != "fp32":
            engine = "x86" if "x86" in torch.backends.quantized.supported_engines else "fbgemm"
            if engine not in torch.backends.quantized.supported_engines:
                raise RuntimeError("Quantized bundle needs a compatible x86 CPU backend.")
            torch.backends.quantized.engine = engine
            specs = {
                torch.nn.Linear: default_dynamic_qconfig,
                torch.nn.Embedding: float_qparams_weight_only_qconfig,
            }
            self.model = quantize_dynamic(self.model, qconfig_spec=specs, inplace=False)
        self.model.load_state_dict(
            torch.load(self.directory / "model_state.pt", map_location="cpu", weights_only=True),
            strict=True,
        )
        self.model.eval()
        self.tokenizer = AutoTokenizer.from_pretrained(self.directory, local_files_only=True)

    def predict(self, resume_text, job_description_text):
        if not isinstance(resume_text, str) or not resume_text.strip():
            raise ValueError("Non-empty extracted CV text is required.")
        if not isinstance(job_description_text, str) or not job_description_text.strip():
            raise ValueError("Non-empty job text is required.")
        enc = self.tokenizer(
            resume_text,
            job_description_text,
            truncation="longest_first",
            max_length=256,
            padding="max_length",
            return_tensors="pt",
        )
        with self.torch.inference_mode():
            prob = self.torch.softmax(self.model(**enc).logits, dim=-1)[0].tolist()
        label_id = max(range(3), key=lambda i: prob[i])
        return dict(
            label_id=label_id,
            label=self.labels[label_id],
            class_scores=dict(zip(self.labels, prob)),
            model_version=self.meta["model_version"],
            input_policy="full text; paired longest_first; max_length=256",
            interpretation="Textual compatibility only; scores are uncalibrated. HR review required.",
        )

def export_cpu_bundle(work, choice="M1", seed_value=42):
    if choice not in ["M0", "M1", "M2"]:
        raise ValueError("Choose a measured model explicitly.")
    if seed_value != 42:
        raise ValueError(
            "Deployment seed is fixed to 42; do not select a seed using test performance."
        )
    p = Path(work) / "public"
    kind = "M1" if choice == "M2" else choice
    info = next(x for x in read(p / f"{kind}_selected.json") if x["seed"] == seed_value)
    m = load_trial(work, info)
    variant = "both" if choice == "M2" else "fp32"
    if variant == "both":
        q = next(
            x
            for x in read(p / "quantization_results.json")
            if x["seed"] == seed_value and x["variant"] == "both"
        )
        assert sha(q["weights"]) == q["weights_sha256"]
        m = quantize(m, "both")
        m.load_state_dict(torch.load(q["weights"], map_location="cpu", weights_only=True))
    token = AutoTokenizer.from_pretrained(STUDENT, revision=STUDENT_REV)
    directory = Path(work) / "private/deployment" / f"{choice}_seed42_{uuid.uuid4().hex[:8]}"
    directory.mkdir(parents=True)
    m.config.save_pretrained(directory)
    token.save_pretrained(directory)
    atomic_save(m.state_dict(), directory / "model_state.pt")
    manifest = dict(
        model_version=directory.name,
        kind=choice,
        variant=variant,
        labels=LABELS,
        max_length=256,
        truncation="longest_first",
        student_revision=STUDENT_REV,
        source_sha256=info["weights_sha256"],
        runtime=environment(),
        status="Research candidate; deployment acceptance and local-versus-LLM comparison remain separate Senior work.",
        files={f.name: sha(f) for f in directory.iterdir() if f.is_file()},
    )
    dump(directory / "model_manifest.json", manifest)
    (directory / "inference.py").write_text(visible_source(CPUFitPredictor))
    (directory / "requirements.txt").write_text(
        f"torch=={torch.__version__.split('+')[0]}\ntransformers=={transformers.__version__}\n"
    )
    # Public synthetic examples only; no benchmark CV text is bundled for deployment.
    examples = [
        ("Python developer with SQL experience.", "Software role requiring Python and SQL."),
        (
            "Customer support and retail experience.",
            "Accountant with financial reporting experience.",
        ),
    ]
    reloaded = CPUFitPredictor(directory)
    for cv, job in examples:
        with torch.inference_mode():
            a = torch.softmax(
                m(
                    **token(
                        cv,
                        job,
                        truncation="longest_first",
                        max_length=256,
                        padding="max_length",
                        return_tensors="pt",
                    )
                ).logits,
                dim=-1,
            )[0].tolist()
        b = list(reloaded.predict(cv, job)["class_scores"].values())
        assert np.allclose(a, b, atol=1e-6, rtol=1e-5)
    dump(
        directory / "roundtrip_check.json",
        dict(
            passed=True,
            examples="Two synthetic text pairs",
            scope="Save/reload consistency, not accuracy or deployment suitability.",
        ),
    )
    print("CPU bundle verified and kept private:", directory)
    return directory

def export_reproducibility(work):
    import zipfile

    w = Path(work)
    p = w / "public"
    folder = p / "reproducibility"
    folder.mkdir(exist_ok=True)
    import sklearn, huggingface_hub, matplotlib

    versions = dict(
        torch=torch.__version__,
        transformers=transformers.__version__,
        numpy=np.__version__,
        pandas=pd.__version__,
        scikit_learn=sklearn.__version__,
        huggingface_hub=huggingface_hub.__version__,
        matplotlib=matplotlib.__version__,
    )
    dump(folder / "tested_versions.json", versions)
    (folder / "requirements.txt").write_text(
        "\n".join(f"{k.replace('_','-')}=={v}" for k, v in versions.items()) + "\n"
    )
    shutil.copy2(ROOT / "splits/splits.csv", folder / "splits.csv")
    (folder / "README.md").write_text(REPRO_README)
    for name, text in [
        ("item3_reviewer.txt", ITEM3_REVIEWER),
        ("item4_reviewer.txt", ITEM4_REVIEWER),
        ("Senior_Integration_Guide.md", INTEGRATION_GUIDE),
    ]:
        (folder / name).write_text(text)
    # These logs are templates. Never invent contribution or human verification entries.
    for name, header in [
        ("individual_contributions.csv", "member,completed_task,evidence_file,date\n"),
        ("ai_assistance_log.csv", "date,tool,purpose,affected_artifact,verification_by_student\n"),
    ]:
        if not (folder / name).exists():
            (folder / name).write_text(header)
    out = w / f'HireMind_public_evidence_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in p.rglob("*"):
            if f.is_file() and f.suffix in [".json", ".csv", ".png", ".pdf", ".md", ".txt"]:
                z.write(f, "results/" + str(f.relative_to(p)))
    print("Public evidence export:", out)
    print(
        "Upload this notebook itself to GitHub too. The final Item 4 ZIP also needs paper, slides and poster; this export is not that final submission."
    )
    return out

## S15 · Guides, reviewer prompts and execution controls
Leave final-test controls disabled until Item 3 is reviewed. Running all setup cells is safe; training requires RUN_TRAINING=True. The optional Item 2 checks can be skipped because historical evidence is included; rerunning them creates new, separately labelled measurements.

In [ ]:
INTEGRATION_GUIDE = r"""# Deep Learning → Senior Integration Guide

## One reusable matching component
The Senior handover describes an interface prototype, sample-data weighted scoring and a separately trained model. FastAPI, PostgreSQL, private CV storage, the queue, real assessments and integration remain planned. The local model is a candidate matching engine, not an established production choice; the handover calls for a comparison against an LLM before adoption.

| Component | Reuse decision | Where it belongs |
|---|---|---|
| Selected BERT-Tiny weights and configuration | Reuse an evaluated checkpoint; export exact version and checksum | Both |
| Tokenizer, label order, paired text formatting | Reuse unchanged: CV first, job second; full-text tokenization; longest-first; combined length 256 | Both |
| CPUFitPredictor and loading checks | Extract the visible class as inference.py; load once at backend startup | Both |
| Quantized M1 | Use only if actual quality, storage and CPU measurements justify it; FP32 remains an option | Both |
| Teacher, optimizer, KD training, seed search, recovery snapshots | Keep out of request-time deployment | DL only |
| Benchmark dataset, split definitions, test protocol, research plots | Research evidence; not the operational database | DL only |
| PDF/DOCX parsing, permissions, queue, file-size/type checks | Implement in the Senior backend; do not alter the scientific preprocessing silently | Senior |
| Application IDs, policy versions, score mapping, ranking, missing-assessment states | Implement and validate in Senior | Senior |
| MCQs, rubric-reviewed practical answers, HR decision, optional LLM feedback | Separate Senior services | Senior |

## Portable bundle
The export contains model_state.pt, config.json, tokenizer files, model_manifest.json, requirements.txt, inference.py and a save/reload check. It contains no CV dataset. Store weights in private deployment storage or an appropriate model-artifact store, not directly in the public repository. The manifest identifies the source checkpoint, labels, input policy, runtime and file checksums.

For FP32 and INT8, use a compatible tested PyTorch/Transformers runtime. The quantized implementation targets supported x86/fbgemm CPU backends; portability does not mean it works on every CPU architecture unchanged. Verify the export on the actual deployment machine.

```python
from inference import CPUFitPredictor
predictor = CPUFitPredictor('/opt/hiremind/models/approved_version')  # once at startup
result = predictor.predict(extracted_cv_text, job_description_text)
```

The response contains label_id, label, class_scores, model_version, input_policy and an interpretation note. Label order is exactly 0=No Fit, 1=Potential Fit, 2=Good Fit. Scores are uncalibrated model outputs, not probabilities of successful employment. No request calls a training function.

## Planned Senior workflow
1. Backend validates the English text-based PDF/DOCX, permissions and size; stores it privately and extracts text. Scanned documents and legacy DOC conversion remain outside the handover pilot.
2. Queue calls the already-loaded matching component with CV text and job text.
3. Save the compatibility prediction against the Application, with model and input-policy versions.
4. A separate scoring service combines approved evidence under a ScoringPolicy. The handover proposes an HR-approved mapping into a component, but that mapping is neither implemented nor validated. Avoid double-counting education/experience and never equate compatibility with demonstrated technical skill.
5. Keep missing required assessments visibly “not assessed”; keep evaluation provisional. Complete approved MCQ and practical-answer assessments.
6. HR reviews evidence and makes the final decision. Keep every application accessible; no automatic rejection or filtering-out solely from this classifier.
7. Optional external LLM drafts feedback; HR reviews it before release. Outcome storage does not automatically retrain the model.

## Changes to make later in Senior
Wrap inference in a backend endpoint; validate the request/response schema; attach application IDs; enforce access and retention; integrate with queue/status updates; record policy/model versions; handle unavailable/corrupt models; measure concurrent end-to-end latency. None requires retraining merely because a CV was uploaded.

Testing 512 tokens, section selection or a different preprocessing policy changes the model input distribution. Treat these as separately labelled controlled experiments with their own evaluation and versions; do not relabel the existing 256-token results or introduce them silently in deployment. An LLM comparison and acceptance criteria belong to Senior engine selection unless explicitly added as separate DL research.

## Acceptance before integration
Verify checksums, CPU load/reload predictions, class order and input contract. Assess Good Fit recall, all per-class errors, latency and memory on the deployment CPU. Compare with the alternative engine on a human-reviewed sample. Define and test scoring mapping and HR access. Dataset licence/provenance and label uncertainty remain limitations despite course approval. A successful export proves portability checks, not hiring validity, fairness or production readiness.
"""
REPRO_README = r"""# HireMind reproducibility
Use the visible-code HireMind_Assignments_2_3_4.ipynb in Google Colab. No hidden implementation files are needed.
Dataset: https://huggingface.co/datasets/cnamuangtoun/resume-job-description-fit at revision 08978e21714984bb417547d2c0f9b477f5298163. Use the checksum-verified frozen split supplied by the notebook. Dataset source/licence/labels remain undocumented; do not redistribute raw CV text.
Run Setup S01-S15, then R01 inventory, R02 preparation, R03 M0, R04 teacher, R05 M1, R06 T=1 ablation, R07 quantization, R08 profiling, R09 tables/error analysis. Training flags are explicit and completed compatible trials are reused. See the notebook for recovery limitations and exact commands.
Item 3 tables and both required plots: public/reports/validation. Raw measurements: public/benchmark_sessions; selected settings: public/*_selected.json; trials and curves: public/trials and public/curves. Keep historical Item 2 scores separate.
After reviewing Item 3, freeze choices in F01 and evaluate held-out test once in F02. Final tables: public/reports/test. Public per-class analyses: public/error_analysis. Private error-review sheets require human interpretation.
Hardware/software are recorded in sessions and training_environment.json, and benchmark specs. Training uses a GPU; deployment profiling uses one CPU thread. Exact PyTorch CUDA-build compatibility is checked before continuing old training. CPU-only evaluation and reporting need no retraining.
Do not upload private/, model weights, credentials or raw CV text. Commit this notebook, code/configuration evidence, package versions, split definitions and generated tables/figures. Add actual contribution and AI assistance records.
Repository: https://github.com/Hams770/HireMind-DL
Portfolio: add the actual instructor-approved portfolio URL if different; do not invent a link.
Item 4 requires a separate final ZIP containing paper PDF, slides PDF/PPTX, A1/A0 poster PDF, README links and AI log if separate. Public evidence export alone is not the final submission. Test these instructions on the actual Colab runtime before claiming full reproducibility.
"""
ITEM3_REVIEWER = r"""Act as a skeptical but constructive reviewer
for a deep learning research paper.

I am working on a CS4085 research project
with two connected contributions:

M0: Original baseline
M1: Baseline + deep learning modification
M2: M1 + efficiency technique

My research question:
[PASTE]

My results table:
[PASTE]

Hardware and experimental settings:
[Device, batch size, precision, threads,
epochs, optimizer, seeds]

My ablation results:
[PASTE]

My interpretation of the results:
[PASTE]

TASK 1 — Results
For each claim, classify it as:
SUPPORTED / WEAK / NOT SUPPORTED.
Explain why.

TASK 2 — Fairness
Identify possible unfair comparisons:
different data splits, training settings,
devices, precision, or measurement methods.

TASK 3 — Reliability
Check whether improvements are consistent
across random seeds.
Identify conclusions that are premature.

TASK 4 — Efficiency
Help explain unexpected findings:
- Fewer parameters but slower inference
- Lower MACs without lower latency
- Quantization that does not improve speed
- Compression that reduces accuracy


TASK 5 — Research Quality
Suggest the three experiments that would
most strengthen the final research paper.

Do not invent results or references.
Do not write my Results section.
Teach me to interpret my own findings.
"""
ITEM4_REVIEWER = r"""Act as a strict reviewer for a deep learning
research workshop.

Our course is CS4085 Deep Learning.
Our presentation track is [A/B/C].

Our research includes:
M0 — Original baseline
M1 — Deep learning modification
M2 — Efficiency technique applied to M1

Here is one section of our paper:
[PASTE]

Our experimental results:
[PASTE]

Review our work using these criteria:

1. Implementation
2. Model Design and Performance
3. Innovation and Relevance
4. Experimental Analysis
5. Scientific Presentation

For each criterion:
- Rate it Excellent / Good / Fair / Poor.
- Explain the rating briefly.
- Identify missing or weak evidence.

Then:

TASK 1 — Scientific Claims
Quote claims that are stronger than the
available evidence. Explain why.

TASK 2 — Numerical Consistency
List results and numerical claims that
must be checked against our tables.

TASK 3 — Experimental Reliability
Identify weaknesses in fairness,
reproducibility, ablations, and statistical
evidence.

TASK 4 — Research Improvement
Suggest the three most important changes
before final submission.


TASK 5 — Oral Defense Practice
Ask each team member two challenging
questions about their contribution and
the overall research.

Wait for our answers before evaluating them.

Do not invent references, results, or evidence.
Do not rewrite our paper.
"""
RUN_TRAINING = False
RUN_ITEM2_CHECKS = False
RUN_QUANTIZATION = False
RUN_CPU_PROFILING = False
RUN_REPORTING = False
FREEZE_CONFIRMATION = ''
RUN_FINAL_TEST = False
EXPORT_MODEL = False
EXPORT_CHOICE = 'M1'  # choose only after measured validation review; do not infer INT8 is best


## C01 · Incident audit — run this first after S01–S15
No training, no file moves, no deletion, no API authorization required. Inspect the current environment, original cached Teacher targets, completed model files and recoverable snapshots. A cached-target PASS does not recreate the Teacher weights.


In [ ]:
incident_inventory = recovery_audit(WORK)


## C02 · Verify cloud storage — run only after reviewing C01
Authorize the SAME Google account as your mounted Drive. This uploads one tiny probe to the project folder and checks its server checksum and reload. It does not purchase storage or start training. Stop if authentication, quota or mount visibility fails; do not weaken the check.


In [ ]:
storage_probe()


## C03 · Authorize reviewed student recovery — empty by default
After inspecting C01 and restoring any usable Bin files, enter only reviewed M1 tags below. `RESUME_TAGS` requires a valid restored snapshot and compatible GPU environment. `RESTART_TAGS` explicitly allows repeating a missing trial when recovery is unavailable. Neither list runs training itself. Records are archived, not silently overwritten. Teacher records/targets are never replaced here.

If Python differs only in its patch version, first discuss the limitation; explicitly set `ALLOW_PYTHON_PATCH_CHANGE=True` only if accepting and documenting that exception. Keep it False otherwise.


In [ ]:
# Leave empty: all restored/repeated M1 trials in this cohort are complete.
RESUME_TAGS = []
RESTART_TAGS = []
for tag in RESUME_TAGS:
    authorize_trial_repair(WORK, tag, allow_restart=False)
for tag in RESTART_TAGS:
    authorize_trial_repair(WORK, tag, allow_restart=True)
if not RESUME_TAGS and not RESTART_TAGS:
    print('No new training authorized. Completed trials remain reusable.')


## R01 · Inspect saved work first — no training
This is the evidence needed to determine what finished in your 18-minute Teacher run. A best-weight file alone is not completion. If the legacy folder is absent, check the Drive account/path before allowing training. The inventory checks files on **your** Drive; the assistant has not inspected them remotely.

In [ ]:
setup(WORK)
inventory = status_inventory(WORK)


## R02 · Prepare/reuse data — no GPU training
Reconstruct the exact frozen training and validation rows, verify labels/IDs, tokenize at 256 and validate caches. The test CSV is not downloaded here. Label-audit sheets remain private and are not filled automatically.

In [ ]:
frames = prepare(WORK)
engineering_dir = WORK / 'private/engineering_checks' / json_hash(environment())
if not (engineering_dir / 'public/engineering_smoke_test.json').exists():
    previous_threads = torch.get_num_threads()
    try:
        smoke(engineering_dir)
    finally:
        torch.set_num_threads(previous_threads)
print('Training and validation preparation complete; test remains sealed.')
if RUN_ITEM2_CHECKS:
    print(baseline_references(frames, WORK))
    print(architecture_and_lengths(frames, WORK))
    print(overfit_check(WORK))
else:
    print('Optional reproduction of Item 2 references/overfit/cost skipped. Historical evidence retained.')


## R03 · M0
Four search trials, then selected seeds 43/44. Valid completed legacy trials are reused without training.
Set RUN_TRAINING=True in S15 when ready, then run this cell. Full recovery saves every validation checkpoint (100 updates and the final update).

In [ ]:
if RUN_TRAINING:
    train_m0(WORK)
else:
    print('Training disabled. Set RUN_TRAINING=True after inspecting R01.')

## R04 · Reuse the verified Teacher and original targets

This stage validates the original cached targets and resolves the recovered Teacher through its separate record and equivalence evidence. It preserves the original missing-file record for provenance. It does not retrain the Teacher. `R01` and `C01` now report `COMPLETE RECOVERED` only after live file, checksum and evidence checks pass.


In [ ]:
if RUN_TRAINING:
    train_teacher(WORK)
else:
    print('Training disabled. Set RUN_TRAINING=True after inspecting R01.')

## R05 · M1 distillation
Four LR×T trials, then selected seeds 43/44. Same student and training budget as M0.
Set RUN_TRAINING=True in S15 when ready, then run this cell. Full recovery saves every validation checkpoint (100 updates and the final update).

In [ ]:
if RUN_TRAINING:
    train_m1(WORK)
else:
    print('Training disabled. Set RUN_TRAINING=True after inspecting R01.')

## R06 · T=1 ablation
The planned E4b uses the selected M1 learning rate and changes only temperature, across three seeds.
Set RUN_TRAINING=True in S15 when ready, then run this cell. Full recovery saves every validation checkpoint (100 updates and the final update).

In [ ]:
if RUN_TRAINING:
    temperature_ablation(WORK)
else:
    print('Training disabled. Set RUN_TRAINING=True after inspecting R01.')

## R07 · M2 and quantization ablations — no training
Set RUN_QUANTIZATION=True after M1 is complete. Linear-only, embeddings-only and combined variants use each selected M1 checkpoint. Existing validated continuation outputs are reused. Prediction agreement with M1 is recorded against the proposal’s 95% pilot check; it is not assumed. If embedding quantization fails, retain the error and report the planned M2/H2 as untestable; do not rename Linear-only quantization as the planned M2. Older quantization measurements may be regenerated because they lack the new source-digest records; no GPU training is repeated.

In [ ]:
if RUN_QUANTIZATION:
    quantize_runs(WORK)
else:
    print('Quantization disabled.')

## R08 · CPU profiling — no training
Set RUN_CPU_PROFILING=True. Run without competing work. All models use the same CPU, one thread, 256 tokens, batch one, 30 warmups and 200 calls on 20 fixed examples. Different runtime/CPU produces a new complete measurement session; do not mix earlier timing numbers. Prior benchmark sessions are retained. Peak RSS includes loading/conversion, not isolated inference memory.

A fresh Python process receives the exact visible function definitions via `python -c`. There is no encoded or hidden implementation file. This isolates benchmarking from the GPU training process.

In [ ]:
if RUN_CPU_PROFILING:
    run_benchmarks(WORK)
else:
    print('CPU profiling disabled.')

## R09 · Item 3 tables, figures and validation error analysis — no training
Set RUN_REPORTING=True once required inputs exist. Produces per-seed accuracy/F1, paired differences, parameter/size/MAC/latency/memory tables, accuracy–latency and quantization-ablation figures, class reports and confusion matrices. Error-review CSVs with raw text stay private. Interpret results yourself; automatic plots do not establish causes.

In [ ]:
if RUN_REPORTING:
    validation_table = comparison_tables(WORK)
    training_diagnostics(WORK)
    error_analysis(WORK, frames, 'validation')
    summarize_label_audit(WORK)
else:
    print('Reporting disabled.')

## H01 · Human review files (CPU; optional)
Each author independently labels the same 90 pairs in their own existing sheet. Keep the answer key and the other author's sheet hidden until both finish. Preserve row_uid/order and use exactly No Fit, Potential Fit or Good Fit. Record brief reasons; do not ask an AI to fill human labels. This is a post-hoc audit, not the originally planned pre-training audit. Do not upload CV text or these sheets to GitHub.

Run the following cell only to locate files. Open the two sheets in Drive and share each with its assigned author. After both return their completed sheets, preserve a backup, replace the respective private CSVs with reviewed files, and run `summarize_label_audit(WORK)`; it will reject missing/duplicate rows. Genuine human error explanations are also required before the report is final.


In [ ]:
for name in ['label_audit_Rawan.csv', 'label_audit_Hams.csv']:
    path = Path(WORK) / 'private' / name
    print(name, '| exists:', path.is_file(), '|', path)
print('No labels changed, downloaded or shared.')


## R10 · Item 3 reviewer and readiness
Inspect each seed, teacher quality and Good Fit recall. Include the measured latency IQR. Review T=1 results as well as quantization ablations. Do not claim significance from three seeds or speed savings from file size. Keep test hypotheses provisional. Run the following exact reviewer prompt with actual tables, hardware, ablation results and your own interpretation. It must critique your work rather than invent results or write your Results section.

In [ ]:
print(ITEM3_REVIEWER)
status_inventory(WORK)

# Assignment 4 — run only after Item 3 is verified
The handout asks for completed planned experiments and error analysis, not a new unrelated method. Our approved proposal additionally reserves the held-out test for final evaluation. Reuse valid M0/M1/M2 weights; **test evaluation does not train models**. E4a and E4b are implemented above. E4c (M0 quantization) remains explicitly optional, as in Item 2; 512-token/section-selection experiments are separate future work, not silently inserted here.

Track A remains Technical Report: use the ACL Overleaf template, maximum 10 pages, and a 150–250-word abstract. The overall guidelines require a portfolio URL as well as GitHub. The reported two-person approval remains in effect. The course AI limit is 25%; this substantial AI-assisted restructuring must be disclosed and reviewed against that rule—rephrasing does not establish compliance.

Paper sections: title/abstract, introduction, related work, methodology, results, discussion, limitations, conclusion, verified references, individual contributions and AI Assistance Log. Use the latest nine-paper Item 2 review; do not replace it with the five-paper proposal review. Check every numerical claim against measured files. Human error analysis, paper writing, presentation/poster and individual defense remain required.

## F01 · Freeze final choices
Only after reviewing Item 3, set FREEZE_CONFIRMATION='FREEZE AFTER ITEM 3 REVIEW' in S15. This records checkpoint/selection hashes and locks further training in this cohort. It does not choose a model from the test set. Export seed is predeclared as 42, not the best test seed.

In [ ]:
if FREEZE_CONFIRMATION:
    freeze_final(WORK, FREEZE_CONFIRMATION)
else:
    print('Final-test gate stays closed.')

## F02 · Held-out final evaluation, error analysis and H1/H2
Set RUN_FINAL_TEST=True only after F01. If interrupted, completed test records are reused; no tuning or training follows test inspection. Ablations remain validation analyses; final M0/M1/M2 test metrics use all three seeds.

In [ ]:
if RUN_FINAL_TEST:
    test_frames = final_test(WORK)
    final_table = comparison_tables(WORK, 'test')
    error_analysis(WORK, test_frames, 'test')
    print(final_hypotheses(WORK))
else:
    print('Held-out test not evaluated.')

## F03 · Final reviewer and submission checklist
Run this prompt against each relevant paper section and actual experimental evidence. Do not invent reviewer ratings or answers. Each student answers defense questions individually. A completed notebook does not replace the paper, slides or poster.

In [ ]:
print(ITEM4_REVIEWER)

## D01 · Export a reusable CPU matching component
Set EXPORT_MODEL=True and choose M0/M1/M2 based on reviewed validation evidence and deployment requirements. M2 is not automatically selected because it is smaller. The export includes visible inference code, local tokenizer/configuration, labels, weights, checksums and tested runtime versions. No Colab import is used by CPUFitPredictor. The private bundle is round-trip checked using synthetic public text; this is not a new accuracy experiment.

In [ ]:
if EXPORT_MODEL:
    deployment_directory = export_cpu_bundle(WORK, EXPORT_CHOICE, 42)
else:
    print('Model export disabled; deployment suitability is not assumed.')

## Deep Learning → Senior Integration Guide
# Deep Learning → Senior Integration Guide

## One reusable matching component
The Senior handover describes an interface prototype, sample-data weighted scoring and a separately trained model. FastAPI, PostgreSQL, private CV storage, the queue, real assessments and integration remain planned. The local model is a candidate matching engine, not an established production choice; the handover calls for a comparison against an LLM before adoption.

| Component | Reuse decision | Where it belongs |
|---|---|---|
| Selected BERT-Tiny weights and configuration | Reuse an evaluated checkpoint; export exact version and checksum | Both |
| Tokenizer, label order, paired text formatting | Reuse unchanged: CV first, job second; full-text tokenization; longest-first; combined length 256 | Both |
| CPUFitPredictor and loading checks | Extract the visible class as inference.py; load once at backend startup | Both |
| Quantized M1 | Use only if actual quality, storage and CPU measurements justify it; FP32 remains an option | Both |
| Teacher, optimizer, KD training, seed search, recovery snapshots | Keep out of request-time deployment | DL only |
| Benchmark dataset, split definitions, test protocol, research plots | Research evidence; not the operational database | DL only |
| PDF/DOCX parsing, permissions, queue, file-size/type checks | Implement in the Senior backend; do not alter the scientific preprocessing silently | Senior |
| Application IDs, policy versions, score mapping, ranking, missing-assessment states | Implement and validate in Senior | Senior |
| MCQs, rubric-reviewed practical answers, HR decision, optional LLM feedback | Separate Senior services | Senior |

## Portable bundle
The export contains model_state.pt, config.json, tokenizer files, model_manifest.json, requirements.txt, inference.py and a save/reload check. It contains no CV dataset. Store weights in private deployment storage or an appropriate model-artifact store, not directly in the public repository. The manifest identifies the source checkpoint, labels, input policy, runtime and file checksums.

For FP32 and INT8, use a compatible tested PyTorch/Transformers runtime. The quantized implementation targets supported x86/fbgemm CPU backends; portability does not mean it works on every CPU architecture unchanged. Verify the export on the actual deployment machine.

```python
from inference import CPUFitPredictor
predictor = CPUFitPredictor('/opt/hiremind/models/approved_version')  # once at startup
result = predictor.predict(extracted_cv_text, job_description_text)
```

The response contains label_id, label, class_scores, model_version, input_policy and an interpretation note. Label order is exactly 0=No Fit, 1=Potential Fit, 2=Good Fit. Scores are uncalibrated model outputs, not probabilities of successful employment. No request calls a training function.

## Planned Senior workflow
1. Backend validates the English text-based PDF/DOCX, permissions and size; stores it privately and extracts text. Scanned documents and legacy DOC conversion remain outside the handover pilot.
2. Queue calls the already-loaded matching component with CV text and job text.
3. Save the compatibility prediction against the Application, with model and input-policy versions.
4. A separate scoring service combines approved evidence under a ScoringPolicy. The handover proposes an HR-approved mapping into a component, but that mapping is neither implemented nor validated. Avoid double-counting education/experience and never equate compatibility with demonstrated technical skill.
5. Keep missing required assessments visibly “not assessed”; keep evaluation provisional. Complete approved MCQ and practical-answer assessments.
6. HR reviews evidence and makes the final decision. Keep every application accessible; no automatic rejection or filtering-out solely from this classifier.
7. Optional external LLM drafts feedback; HR reviews it before release. Outcome storage does not automatically retrain the model.

## Changes to make later in Senior
Wrap inference in a backend endpoint; validate the request/response schema; attach application IDs; enforce access and retention; integrate with queue/status updates; record policy/model versions; handle unavailable/corrupt models; measure concurrent end-to-end latency. None requires retraining merely because a CV was uploaded.

Testing 512 tokens, section selection or a different preprocessing policy changes the model input distribution. Treat these as separately labelled controlled experiments with their own evaluation and versions; do not relabel the existing 256-token results or introduce them silently in deployment. An LLM comparison and acceptance criteria belong to Senior engine selection unless explicitly added as separate DL research.

## Acceptance before integration
Verify checksums, CPU load/reload predictions, class order and input contract. Assess Good Fit recall, all per-class errors, latency and memory on the deployment CPU. Compare with the alternative engine on a human-reviewed sample. Define and test scoring mapping and HR access. Dataset licence/provenance and label uncertainty remain limitations despite course approval. A successful export proves portability checks, not hiring validity, fairness or production readiness.


## D02 · Export public reproducibility evidence
Run after generating the evidence you need. This ZIP excludes raw CV text, model weights, teacher targets, recovery snapshots and private human-audit sheets. Upload this single notebook itself to GitHub as the code artifact, alongside configurations/results. The public evidence ZIP is **not** Assignment 4's final paper/slides/poster ZIP.

In [ ]:
public_zip = export_reproducibility(WORK)
# To download after checking the file list:
# from google.colab import files
# files.download(str(public_zip))

# Appendix · Original Assignment 2 code, preserved for traceability
The original code below is kept **readable and unchanged** as an archival reference, including its old environment and paths. Its active functionality has been integrated above. These are Markdown code blocks deliberately excluded from Run all so you do not retrain the old baseline or overwrite its outputs by accident. Original saved results are in HISTORICAL_ITEM2; the original uploaded notebook remains unchanged. The new controlled cohort uses the reviewed Item 3 protocol (including validation batch 16) and never relabels historical Item 2 results as new results.

### Original Item 2 cell 2
```python
# Colab already has torch, transformers, scikit-learn and matplotlib; nothing is upgraded, so no restart is needed.
!pip -q install huggingface_hub
```

### Original Item 2 cell 3
```python
%%writefile data_prep.py
"""HireMind DL - dataset audit, cleaning and grouped train/validation/test split.

Pure pandas / scikit-learn (no PyTorch), so it can be tested anywhere.
Used by notebooks/HireMind_M0_Baseline.ipynb.

Split policy (fixed before any model training):
  * IDs: cv_id / job_id = first 16 hex chars of SHA-256 of whitespace-normalised, lower-cased text.
  * Near-duplicate job descriptions (TF-IDF cosine >= NEAR_DUP_THRESHOLD) share one job_group.
  * Every row of a pair that carries conflicting labels is removed (any split).
  * Exact duplicate pairs: one copy is kept; if a pair occurs in both files, the training copy is dropped.
  * Training rows whose job is a near-duplicate of a test job are dropped.
  * The official test file is kept as the test set (only the removals above).
  * Validation = one fold of StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    on the cleaned training file, grouped by job_group, stratified by label.
"""
import hashlib
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedGroupKFold

LABELS = ["No Fit", "Potential Fit", "Good Fit"]
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
NEAR_DUP_THRESHOLD = 0.95


def norm(text):
    return re.sub(r"\s+", " ", str(text)).strip().lower()


def text_id(text):
    return hashlib.sha256(norm(text).encode("utf-8")).hexdigest()[:16]


def add_ids(train_df, test_df):
    """Return one frame with split, orig_index, cv_id, job_id, pair_id, label_id."""
    frames = []
    for name, df in (("train", train_df), ("test", test_df)):
        d = df[["resume_text", "job_description_text", "label"]].copy()
        d["orig_split"] = name
        d["orig_index"] = np.arange(len(d))
        frames.append(d)
    data = pd.concat(frames, ignore_index=True)
    unknown = set(data["label"]) - set(LABELS)
    if unknown:
        raise ValueError(f"Unexpected labels: {unknown}")
    data["cv_id"] = data["resume_text"].map(text_id)
    data["job_id"] = data["job_description_text"].map(text_id)
    data["pair_id"] = data["cv_id"] + "_" + data["job_id"]
    data["label_id"] = data["label"].map(LABEL2ID)
    data["row_uid"] = data["orig_split"] + "_" + data["orig_index"].astype(str)
    return data


def near_duplicate_groups(texts_by_id, threshold=NEAR_DUP_THRESHOLD):
    """Union-find over TF-IDF cosine similarity. Returns {id: group_id} and the list of near-dup pairs."""
    ids = list(texts_by_id.keys())
    if len(ids) < 2:
        return {i: i for i in ids}, []
    tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    matrix = tfidf.fit_transform([texts_by_id[i] for i in ids])
    sim = cosine_similarity(matrix)
    np.fill_diagonal(sim, 0.0)
    parent = list(range(len(ids)))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    pairs = []
    rows, cols = np.where(np.triu(sim, k=1) >= threshold)
    for r, c in zip(rows, cols):
        pairs.append((ids[r], ids[c], float(sim[r, c])))
        parent[find(r)] = find(c)
    groups = {ids[i]: ids[find(i)] for i in range(len(ids))}
    return groups, pairs


def audit(data):
    """Dataset statistics before cleaning."""
    stats = {"rows": int(len(data))}
    for split in ("train", "test"):
        d = data[data.orig_split == split]
        stats[split] = {
            "rows": int(len(d)),
            "class_counts": {k: int(v) for k, v in Counter(d["label"]).items()},
            "unique_cvs": int(d.cv_id.nunique()),
            "unique_jobs": int(d.job_id.nunique()),
        }
    stats["class_counts_total"] = {k: int(v) for k, v in Counter(data["label"]).items()}
    stats["unique_cvs"] = int(data.cv_id.nunique())
    stats["unique_jobs"] = int(data.job_id.nunique())
    labels_per_pair = data.groupby("pair_id")["label"].nunique()
    conflicting = labels_per_pair[labels_per_pair > 1].index
    stats["unique_pairs"] = int(data.pair_id.nunique())
    stats["duplicate_pair_excess_rows"] = int(len(data) - data.pair_id.nunique())
    stats["conflicting_pairs"] = int(len(conflicting))
    stats["conflicting_rows"] = int(data.pair_id.isin(conflicting).sum())
    tr, te = data[data.orig_split == "train"], data[data.orig_split == "test"]
    stats["overlap_train_test"] = {
        "cvs": int(len(set(tr.cv_id) & set(te.cv_id))),
        "jobs": int(len(set(tr.job_id) & set(te.job_id))),
        "pairs": int(len(set(tr.pair_id) & set(te.pair_id))),
    }
    return stats, set(conflicting)


def clean(data, conflicting_pairs):
    """Apply the removal rules; return cleaned frame and a removal log."""
    log = {}
    before = data.orig_split.value_counts().to_dict()
    d = data[~data.pair_id.isin(conflicting_pairs)].copy()
    log["removed_conflicting_rows"] = {
        s: int(before.get(s, 0) - (d.orig_split == s).sum()) for s in ("train", "test")
    }
    test_pairs = set(d.loc[d.orig_split == "test", "pair_id"])
    mask_cross = (d.orig_split == "train") & d.pair_id.isin(test_pairs)
    log["removed_train_rows_duplicated_in_test"] = int(mask_cross.sum())
    d = d[~mask_cross]
    n0 = d.orig_split.value_counts().to_dict()
    d = d.drop_duplicates(subset=["orig_split", "pair_id"], keep="first")
    log["removed_within_split_duplicates"] = {
        s: int(n0.get(s, 0) - (d.orig_split == s).sum()) for s in ("train", "test")
    }
    return d.reset_index(drop=True), log


def remove_train_rows_sharing_test_job_group(d, job_groups):
    """Drop training rows whose job is a near-duplicate of a test job (prevents job-level leakage)."""
    test_groups = {job_groups[j] for j in d.loc[d.orig_split == "test", "job_id"]}
    mask = (d.orig_split == "train") & d.job_id.map(job_groups).isin(test_groups)
    return d[~mask].reset_index(drop=True), int(mask.sum())


def make_splits(d, job_groups, seed=42, n_splits=5):
    """Grouped, stratified validation split from the cleaned training file."""
    d = d.copy()
    d["job_group"] = d["job_id"].map(job_groups)
    tr = d[d.orig_split == "train"]
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    _, val_idx = next(sgkf.split(tr, tr["label_id"], groups=tr["job_group"]))
    val_rows = set(tr.index[val_idx])
    d["split"] = np.where(d.orig_split == "test", "test",
                          np.where(d.index.isin(val_rows), "validation", "train"))
    return d


def check_splits(d):
    """Assertions that the split policy holds. Returns a dict of checks."""
    s = {k: d[d.split == k] for k in ("train", "validation", "test")}
    checks = {
        "train_val_job_group_overlap": len(set(s["train"].job_group) & set(s["validation"].job_group)),
        "train_test_job_overlap": len(set(s["train"].job_id) & set(s["test"].job_id)),
        "val_test_job_overlap": len(set(s["validation"].job_id) & set(s["test"].job_id)),
        "pair_overlap_any": len(set(s["train"].pair_id) & set(s["validation"].pair_id))
        + len(set(s["train"].pair_id) & set(s["test"].pair_id))
        + len(set(s["validation"].pair_id) & set(s["test"].pair_id)),
        "classes_in_validation": int(s["validation"].label_id.nunique()),
        "cv_overlap_train_test": len(set(s["train"].cv_id) & set(s["test"].cv_id)),
        "cv_overlap_train_val": len(set(s["train"].cv_id) & set(s["validation"].cv_id)),
    }
    assert checks["train_val_job_group_overlap"] == 0, "job group leaks between train and validation"
    assert checks["pair_overlap_any"] == 0, "identical pair in two splits"
    assert checks["classes_in_validation"] == 3, "validation is missing a class"
    return checks


def split_counts(d):
    out = {}
    for k in ("train", "validation", "test"):
        part = d[d.split == k]
        out[k] = {
            "rows": int(len(part)),
            "cvs": int(part.cv_id.nunique()),
            "jobs": int(part.job_id.nunique()),
            "class_counts": {LABELS[i]: int((part.label_id == i).sum()) for i in range(3)},
        }
    return out


def save_split_definition(d, path):
    """Save IDs only (no CV or job text is written)."""
    cols = ["row_uid", "orig_split", "orig_index", "split", "cv_id", "job_id", "job_group", "pair_id", "label", "label_id"]
    d[cols].to_csv(path, index=False)


if __name__ == "__main__":
    print(json.dumps({"labels": LABELS, "near_dup_threshold": NEAR_DUP_THRESHOLD}))
```

### Original Item 2 cell 4
```python
%%writefile bench_cpu.py
"""CPU latency and memory benchmark for a saved BERT-Tiny classifier state_dict.

Runs in a fresh process so peak memory is not polluted by training.
Usage:
  python bench_cpu.py --weights m0.pt --example example.json --out bench.json
example.json = {"resume": "...", "job": "..."}  (one validation pair, kept outside the repo)
"""
import argparse
import json
import statistics
import time

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

STUDENT = "google/bert_uncased_L-2_H-128_A-2"


def mem_mib(field):
    """Read VmRSS (current) or VmHWM (peak) of THIS process from /proc/self/status.
    ru_maxrss is not used: on Linux it can carry over the parent notebook's peak after exec."""
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith(field + ":"):
                return int(line.split()[1]) / 1024.0  # kB -> MiB
    return float("nan")


def timed(fn, warmup, runs):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000.0)
    times.sort()
    return {"median_ms": statistics.median(times), "p95_ms": times[int(0.95 * (len(times) - 1))]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--example", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--threads", type=int, default=1)
    ap.add_argument("--max_len", type=int, default=256)
    ap.add_argument("--max_chars", type=int, default=None)  # None = full texts, as in training
    ap.add_argument("--revision", default=None)             # pinned model/tokenizer commit
    ap.add_argument("--warmup", type=int, default=30)
    ap.add_argument("--runs", type=int, default=200)
    a = ap.parse_args()

    torch.set_num_threads(a.threads)
    rss_after_imports = mem_mib("VmRSS")
    tok = AutoTokenizer.from_pretrained(STUDENT, revision=a.revision)
    model = AutoModelForSequenceClassification.from_pretrained(STUDENT, revision=a.revision, num_labels=3, attn_implementation="eager")
    model.load_state_dict(torch.load(a.weights, map_location="cpu"))
    model.eval()
    ex = json.load(open(a.example))

    def encode():
        return tok(ex["resume"][:a.max_chars], ex["job"][:a.max_chars], truncation="longest_first", max_length=a.max_len,
                   padding="max_length", return_tensors="pt")

    enc = encode()
    with torch.inference_mode():
        model_only = timed(lambda: model(**enc), a.warmup, a.runs)
        with_tok = timed(lambda: model(**encode()), a.warmup, a.runs)
    result = {
        "threads": a.threads, "batch_size": 1, "seq_len": a.max_len,
        "warmup": a.warmup, "runs": a.runs, "max_chars": a.max_chars, "revision": a.revision,
        "model_only": model_only, "tokenization_inclusive": with_tok,
        "rss_after_imports_mib": rss_after_imports, "peak_rss_mib": mem_mib("VmHWM"),
        "rss_end_mib": mem_mib("VmRSS"),
        "torch": torch.__version__,
    }
    json.dump(result, open(a.out, "w"), indent=2)
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()
```

### Original Item 2 cell 5
```python
import os, json, time, random, platform, hashlib, statistics, copy, shutil, subprocess, sys
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import transformers, sklearn
import huggingface_hub
from huggingface_hub import list_repo_files, hf_hub_download, model_info
import importlib, data_prep as dp
importlib.reload(dp)

SEED = 42
REPO = "cnamuangtoun/resume-job-description-fit"
REVISION = "08978e21714984bb417547d2c0f9b477f5298163"   # pinned revision from Assignment 1
EXPECTED_SHA = {"train": "b86fa488a21258015de1750633a9ac410f701ff96bf533f68978dbe3adec6d40",
                "test":  "f3c48f4cc3559bf192ebb9c241a30c601b5b8b3eee6f1656dd4d26dfd4b0887a"}
STUDENT = "google/bert_uncased_L-2_H-128_A-2"
STUDENT_REV = model_info(STUDENT).sha     # exact model/tokenizer commit; every load below uses it (M1/M2 reuse this value)
MAX_CHARS = None                         # no character cut: full texts are tokenised (a 6,000-char cut changed 65 of 6,155 inputs)
MAX_LEN, BATCH, EPOCHS = 256, 16, 3
LRS = [2e-5, 5e-5, 1e-4, 3e-4]          # M0 search space (4 trials, as planned)
EXTRA_SEEDS = [43, 44]
EVAL_EVERY = 100                          # validation evaluation interval (steps)
OUT = "outputs"
for sub in ["figures", "results", "splits", "tex"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
os.makedirs("private", exist_ok=True)     # never zipped: contains raw text

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Deterministic GPU kernels so a rerun gives the same numbers (warn_only: never stops the run)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def cpu_name():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "unknown"

ENV = {"python": platform.python_version(), "torch": torch.__version__,
       "transformers": transformers.__version__, "sklearn": sklearn.__version__,
       "device": DEVICE, "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none",
       "numpy": np.__version__, "pandas": pd.__version__, "huggingface_hub": huggingface_hub.__version__,
       "student_model": STUDENT, "student_revision": STUDENT_REV, "dataset_revision": REVISION,
       "cpu": cpu_name(), "cpu_count": os.cpu_count()}
print(json.dumps(ENV, indent=2))
with open(f"{OUT}/results/tested_versions.txt", "w") as f:   # versions this run was tested with
    for k in ("python", "torch", "transformers", "sklearn", "numpy", "pandas", "huggingface_hub"):
        f.write(f"{k}=={ENV[k]}\n")
    f.write(f"# {STUDENT} @ {STUDENT_REV}\n# dataset revision {REVISION}\n")
```

### Original Item 2 cell 7
```python
files = list_repo_files(REPO, repo_type="dataset", revision=REVISION)
print("Files in repo:", files)
paths = {}
for f in files:
    if f.lower().endswith(".csv"):
        key = "train" if "train" in f.lower() else ("test" if "test" in f.lower() else f)
        paths[key] = hf_hub_download(REPO, f, repo_type="dataset", revision=REVISION)
sha = {k: hashlib.sha256(open(p, "rb").read()).hexdigest() for k, p in paths.items()}
for k in ("train", "test"):
    assert sha.get(k) == EXPECTED_SHA[k], f"{k} file hash differs from the Assignment 1 audit - stop"
    print(k, "hash matches Assignment 1 audit")
train_df = pd.read_csv(paths["train"])
test_df = pd.read_csv(paths["test"])
print(train_df.shape, test_df.shape, list(train_df.columns))
```

### Original Item 2 cell 9
```python
data = dp.add_ids(train_df, test_df)
stats, conflicting = dp.audit(data)

# Near-duplicate check (TF-IDF cosine >= 0.95) for job descriptions and CVs
job_texts = dict(zip(data.job_id, data.job_description_text))
cv_texts = dict(zip(data.cv_id, data.resume_text))
job_groups, job_pairs = dp.near_duplicate_groups(job_texts)
cv_groups, cv_pairs = dp.near_duplicate_groups(cv_texts)
train_jobs = set(data.loc[data.orig_split == "train", "job_id"]); test_jobs = set(data.loc[data.orig_split == "test", "job_id"])
cross_job_pairs = [p for p in job_pairs if (p[0] in train_jobs and p[1] in test_jobs) or (p[1] in train_jobs and p[0] in test_jobs)]
stats["near_duplicates"] = {
    "threshold": dp.NEAR_DUP_THRESHOLD,
    "job_pairs": len(job_pairs), "job_pairs_across_train_test": len(cross_job_pairs),
    "job_groups": len(set(job_groups.values())),
    "cv_pairs": len(cv_pairs), "cv_groups": len(set(cv_groups.values())),
}
stats["source"] = {"repo": REPO, "revision": REVISION, "files": files, "sha256": sha,
                   "licence": "none stated in repository (no dataset card)",
                   "instructor_approved": True}
print(json.dumps(stats, indent=2))
```

### Original Item 2 cell 10
```python
# Token lengths with the student tokenizer (to see how much 256-token truncation removes)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(STUDENT, revision=STUDENT_REV)
def n_tokens(texts):
    return [len(x) for x in tok(list(texts), add_special_tokens=False)["input_ids"]]
cv_len = dict(zip(cv_texts.keys(), n_tokens(cv_texts.values())))
job_len = dict(zip(job_texts.keys(), n_tokens(job_texts.values())))
data["pair_tokens"] = data.cv_id.map(cv_len) + data.job_id.map(job_len) + 3   # [CLS] cv [SEP] job [SEP]
stats["tokens"] = {
    "cv_median": float(np.median(list(cv_len.values()))), "job_median": float(np.median(list(job_len.values()))),
    "pair_median": float(data.pair_tokens.median()),
    "pair_p90": float(data.pair_tokens.quantile(0.9)),
    "pct_pairs_truncated_at_256": float(100 * (data.pair_tokens > MAX_LEN).mean()),
}
print(stats["tokens"])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
counts = data.groupby(["orig_split", "label"]).size().unstack()[dp.LABELS]
counts.plot.bar(ax=ax[0], rot=0); ax[0].set_title("Class counts per original file"); ax[0].set_xlabel(""); ax[0].set_ylabel("pairs")
ax[1].hist(data.pair_tokens, bins=60, color="#4C72B0"); ax[1].axvline(MAX_LEN, color="red", ls="--", label=f"max length {MAX_LEN}")
ax[1].set_title("CV + job length (BERT word-pieces)"); ax[1].set_xlabel("tokens per pair"); ax[1].set_ylabel("pairs"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/figures/dataset_overview.png", dpi=200); plt.show()
```

### Original Item 2 cell 12
```python
clean_df, removal_log = dp.clean(data, conflicting)
clean_df, n_near = dp.remove_train_rows_sharing_test_job_group(clean_df, job_groups)
removal_log["removed_train_rows_near_duplicate_of_test_job"] = n_near
splits = dp.make_splits(clean_df, job_groups, seed=SEED)
checks = dp.check_splits(splits)
counts = dp.split_counts(splits)
dp.save_split_definition(splits, f"{OUT}/splits/splits.csv")
manifest = {"seed": SEED, "policy": dp.__doc__.strip(), "removals": removal_log,
            "checks": checks, "counts": counts, "source": stats["source"]}
json.dump(manifest, open(f"{OUT}/splits/manifest.json", "w"), indent=2)
json.dump(stats, open(f"{OUT}/results/dataset_stats.json", "w"), indent=2)
print(json.dumps({"removals": removal_log, "checks": checks, "counts": counts}, indent=2))

tbl = pd.DataFrame({k: v["class_counts"] for k, v in counts.items()}).T[dp.LABELS]
tbl.plot.bar(stacked=True, rot=0, figsize=(6, 3.4)); plt.title("Final split sizes by class"); plt.ylabel("pairs")
plt.tight_layout(); plt.savefig(f"{OUT}/figures/split_classes.png", dpi=200); plt.show()
```

### Original Item 2 cell 13
```python
# Private label-audit sheet (90 training pairs, 30 per class, seed 42) for the planned manual check.
# Contains raw text, so it goes to ./private and is NOT included in the outputs zip or GitHub.
tr_rows = splits[splits.split == "train"]
sample = (tr_rows.groupby("label").sample(n=30, random_state=SEED)
          .sample(frac=1, random_state=SEED).reset_index(drop=True))
sample.insert(0, "audit_id", range(1, len(sample) + 1))
sheet = sample[["audit_id", "resume_text", "job_description_text"]].assign(rawan_label="", hams_label="")
sheet.to_csv("private/label_audit_sheet.csv", index=False)               # labels hidden from annotators
sample[["audit_id", "row_uid", "label"]].to_csv("private/label_audit_key.csv", index=False)
print("Saved private/label_audit_sheet.csv and private/label_audit_key.csv")
```

### Original Item 2 cell 15
```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

tr = splits[splits.split == "train"]; va = splits[splits.split == "validation"]
def scores(y, p):
    return {"macro_f1": 100 * f1_score(y, p, average="macro"), "accuracy": 100 * accuracy_score(y, p)}

majority = int(tr.label_id.mode()[0])
ref = {"majority": scores(va.label_id, np.full(len(va), majority))}
vec = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)
Xtr = vec.fit_transform(tr.resume_text + " [SEP] " + tr.job_description_text)
Xva = vec.transform(va.resume_text + " [SEP] " + va.job_description_text)
logreg = LogisticRegression(max_iter=3000, random_state=SEED).fit(Xtr, tr.label_id)
ref["tfidf_logreg"] = scores(va.label_id, logreg.predict(Xva))
print(json.dumps(ref, indent=2))
```

### Original Item 2 cell 17
```python
from transformers import AutoModelForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(STUDENT, revision=STUDENT_REV)

def new_model(seed, dropout=None):
    set_seed(seed)                       # makes the new classifier head identical for M0 and (later) M1
    kw = {} if dropout is None else {"hidden_dropout_prob": dropout, "attention_probs_dropout_prob": dropout}
    return AutoModelForSequenceClassification.from_pretrained(STUDENT, revision=STUDENT_REV, num_labels=3, attn_implementation="eager", **kw)

import gc
for name in ("Xtr", "Xva", "vec", "logreg", "cv_texts", "job_texts", "cv_len", "job_len"):
    globals().pop(name, None)           # free memory from earlier sections
gc.collect()

def encode(df, chunk=128, max_chars=MAX_CHARS):
    """Tokenise the full texts in small chunks to keep RAM low (no character cut when max_chars is None)."""
    ids, am, tt = [], [], []
    cvs = [t[:max_chars] for t in df.resume_text]; jobs = [t[:max_chars] for t in df.job_description_text]
    for i in range(0, len(cvs), chunk):
        enc = tok(cvs[i:i + chunk], jobs[i:i + chunk], truncation="longest_first",
                  max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        ids.append(enc["input_ids"]); am.append(enc["attention_mask"]); tt.append(enc["token_type_ids"])
    return TensorDataset(torch.cat(ids), torch.cat(am), torch.cat(tt),
                         torch.tensor(df.label_id.values, dtype=torch.long))

train_ds, val_ds = encode(tr), encode(va)     # the test set is NOT encoded in this notebook
print(len(train_ds), len(val_ds))


m = new_model(SEED).eval()
blocks = {"embeddings": m.bert.embeddings, "encoder.layer.0": m.bert.encoder.layer[0],
          "encoder.layer.1": m.bert.encoder.layer[1], "pooler": m.bert.pooler, "classifier": m.classifier}
params = {k: sum(p.numel() for p in b.parameters()) for k, b in blocks.items()}
params["total"] = sum(p.numel() for p in m.parameters())
params["word_embeddings"] = m.bert.embeddings.word_embeddings.weight.numel()

# Measured MACs (FLOPs / 2) for one pair of 256 tokens
from torch.utils.flop_counter import FlopCounterMode
x = {"input_ids": train_ds.tensors[0][:1], "attention_mask": train_ds.tensors[1][:1], "token_type_ids": train_ds.tensors[2][:1]}
with FlopCounterMode(display=False) as fc, torch.no_grad():
    m(**x)
flop_counts = fc.get_flop_counts()
def block_macs(suffix):
    for name, ops in flop_counts.items():
        if name.endswith("." + suffix):
            return sum(ops.values()) / 2
    return 0.0
macs = {k: block_macs(k) for k in blocks}
macs["total"] = fc.get_total_flops() / 2

# Analytic check: per layer 4·L·H² (Q,K,V,O) + 2·L²·H (scores, weighted sum) + 2·L·H·I (feed-forward)
L, H, I = MAX_LEN, m.config.hidden_size, m.config.intermediate_size
analytic_layer = 4 * L * H * H + 2 * L * L * H + 2 * L * H * I
macs_analytic = {"per_layer": analytic_layer, "pooler": H * H, "classifier": H * 3,
                 "total": m.config.num_hidden_layers * analytic_layer + H * H + H * 3}
cost_static = {"params": params, "macs_measured": macs, "macs_analytic": macs_analytic,
               "config": {"layers": m.config.num_hidden_layers, "hidden": H, "heads": m.config.num_attention_heads,
                          "intermediate": I, "vocab": m.config.vocab_size, "max_len": MAX_LEN}}
print(json.dumps(cost_static, indent=2))
```

### Original Item 2 cell 19
```python
set_seed(SEED)
probe = new_model(SEED, dropout=0.0).to(DEVICE).train()      # dropout off only for this sanity check
idx = torch.randperm(len(train_ds), generator=torch.Generator().manual_seed(SEED))[:16]
b = [t[idx].to(DEVICE) for t in train_ds.tensors]
opt = torch.optim.AdamW(probe.parameters(), lr=1e-3)
overfit_losses = []
for step in range(300):
    out = probe(input_ids=b[0], attention_mask=b[1], token_type_ids=b[2], labels=b[3])
    opt.zero_grad(); out.loss.backward(); opt.step()
    overfit_losses.append(out.loss.item())
probe.eval()
with torch.no_grad():
    acc = (probe(input_ids=b[0], attention_mask=b[1], token_type_ids=b[2]).logits.argmax(-1) == b[3]).float().mean().item()
overfit = {"batch_size": 16, "steps": 300, "lr": 1e-3, "first_loss": overfit_losses[0],
           "final_loss": overfit_losses[-1], "final_accuracy": 100 * acc,
           "passed": bool(overfit_losses[-1] < 0.05 and acc == 1.0)}
print(overfit)
plt.figure(figsize=(5, 3)); plt.plot(overfit_losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("training loss")
plt.title("Overfit one batch (16 pairs)"); plt.tight_layout(); plt.savefig(f"{OUT}/figures/overfit_one_batch.png", dpi=200); plt.show()
del probe, opt, b; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()
```

### Original Item 2 cell 21
```python
from transformers import get_linear_schedule_with_warmup

def evaluate(model, ds, bs=128):
    model.eval(); total, n, preds, labels = 0.0, 0, [], []
    with torch.no_grad():
        for ids, am, tt, y in DataLoader(ds, batch_size=bs):
            ids, am, tt, y = ids.to(DEVICE), am.to(DEVICE), tt.to(DEVICE), y.to(DEVICE)
            out = model(input_ids=ids, attention_mask=am, token_type_ids=tt, labels=y)
            total += out.loss.item() * len(y); n += len(y)
            preds.append(out.logits.argmax(-1).cpu()); labels.append(y.cpu())
    p, l = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return {"loss": total / n, **scores(l, p)}, p, l

def train_m0(lr, seed):
    model = new_model(seed).to(DEVICE)
    loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, generator=torch.Generator().manual_seed(seed))
    total_steps = EPOCHS * len(loader)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * total_steps), total_steps)
    hist = {"step": [], "train_loss": [], "val_loss": [], "val_macro_f1": [], "epoch_end_steps": []}
    best, best_state, step, running = {"macro_f1": -1.0}, None, 0, []
    if DEVICE == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        gpu_before = torch.cuda.memory_allocated() / 2**20   # only this run's model should be on the GPU now
    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        for ids, am, tt, y in loader:
            ids, am, tt, y = ids.to(DEVICE), am.to(DEVICE), tt.to(DEVICE), y.to(DEVICE)
            loss = model(input_ids=ids, attention_mask=am, token_type_ids=tt, labels=y).loss
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); running.append(loss.item()); step += 1
            if step % EVAL_EVERY == 0 or step == total_steps:
                v, _, _ = evaluate(model, val_ds); model.train()
                hist["step"].append(step); hist["train_loss"].append(float(np.mean(running))); running = []
                hist["val_loss"].append(v["loss"]); hist["val_macro_f1"].append(v["macro_f1"])
                if v["macro_f1"] > best["macro_f1"]:
                    best = {**v, "step": step, "epoch": epoch + 1}
                    best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        hist["epoch_end_steps"].append(step)
    info = {"lr": lr, "seed": seed, "best": best, "history": hist, "steps": total_steps,
            "train_time_s": time.time() - t0,
            "peak_gpu_mem_mib": torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else None,
            "gpu_mem_before_run_mib": gpu_before if DEVICE == "cuda" else None}
    model.load_state_dict(best_state)
    return model, info

runs, models = [], {}
for lr in LRS:
    model, info = train_m0(lr, SEED)
    runs.append(info); models[lr] = model.cpu()     # move off the GPU so the next run's memory is measured alone
    del model; gc.collect()
    print(f"lr={lr:g}: best val macro-F1 {info['best']['macro_f1']:.2f} (acc {info['best']['accuracy']:.2f}, "
          f"step {info['best']['step']}), {info['train_time_s']:.0f}s")
best_run = max(runs, key=lambda r: (r["best"]["macro_f1"], -r["best"]["loss"]))
BEST_LR = best_run["lr"]
m0 = models[BEST_LR].to(DEVICE)
print("Selected learning rate (validation only):", BEST_LR)
```

### Original Item 2 cell 22
```python
seed_runs = [best_run]
for s in EXTRA_SEEDS:
    m0 = m0.cpu()
    _m, info = train_m0(BEST_LR, s); del _m; gc.collect()
    m0 = m0.to(DEVICE)
    seed_runs.append(info)
    print(f"seed {s}: val macro-F1 {info['best']['macro_f1']:.2f}")
f1s = [r["best"]["macro_f1"] for r in seed_runs]; accs = [r["best"]["accuracy"] for r in seed_runs]
seed_summary = {"seeds": [r["seed"] for r in seed_runs], "val_macro_f1": f1s, "val_accuracy": accs,
                "macro_f1_mean": float(np.mean(f1s)), "macro_f1_sd": float(np.std(f1s, ddof=1)),
                "accuracy_mean": float(np.mean(accs)), "accuracy_sd": float(np.std(accs, ddof=1))}
print(json.dumps(seed_summary, indent=2))
```

### Original Item 2 cell 24
```python
h = best_run["history"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(h["step"], h["train_loss"], label="training loss"); ax[0].plot(h["step"], h["val_loss"], label="validation loss")
for s in h["epoch_end_steps"][:-1]: ax[0].axvline(s, color="grey", ls=":", lw=0.8)
ax[0].set_xlabel("step"); ax[0].set_ylabel("cross-entropy"); ax[0].set_title(f"M0 loss curves (lr={BEST_LR:g}, seed {SEED})"); ax[0].legend()
ax[1].plot(h["step"], h["val_macro_f1"], color="green"); ax[1].axhline(ref["majority"]["macro_f1"], color="red", ls="--", label="majority class")
ax[1].set_xlabel("step"); ax[1].set_ylabel("validation macro-F1 (%)"); ax[1].set_title("Validation macro-F1"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/figures/m0_curves.png", dpi=200); plt.show()

plt.figure(figsize=(5, 3))
plt.bar([f"{r['lr']:g}" for r in runs], [r["best"]["macro_f1"] for r in runs], color="#4C72B0")
plt.xlabel("learning rate"); plt.ylabel("best val macro-F1 (%)"); plt.title("M0 learning-rate search (seed 42)")
plt.tight_layout(); plt.savefig(f"{OUT}/figures/m0_lr_search.png", dpi=200); plt.show()

val_metrics, val_pred, val_true = evaluate(m0, val_ds)
cm = confusion_matrix(val_true, val_pred)
report = classification_report(val_true, val_pred, target_names=dp.LABELS, output_dict=True, digits=4)
fig, axc = plt.subplots(figsize=(4.4, 3.8)); im = axc.imshow(cm, cmap="Blues")
axc.set_xticks(range(3), dp.LABELS, rotation=20); axc.set_yticks(range(3), dp.LABELS)
for i in range(3):
    for j in range(3):
        axc.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
axc.set_xlabel("predicted"); axc.set_ylabel("true"); axc.set_title("M0 validation confusion matrix")
plt.tight_layout(); plt.savefig(f"{OUT}/figures/m0_confusion_val.png", dpi=200); plt.show()
print(json.dumps(report, indent=2))
```

### Original Item 2 cell 26
```python
torch.save({k: v.cpu() for k, v in m0.state_dict().items()}, "private/m0_best.pt")
size_bytes = os.path.getsize("private/m0_best.pt")
ex = va.iloc[0]
json.dump({"resume": ex.resume_text, "job": ex.job_description_text}, open("private/example.json", "w"))
subprocess.run([sys.executable, "bench_cpu.py", "--weights", "private/m0_best.pt",
                "--revision", STUDENT_REV,
                "--example", "private/example.json", "--out", f"{OUT}/results/m0_cpu_bench.json"], check=True)
bench = json.load(open(f"{OUT}/results/m0_cpu_bench.json"))
print("Serialized state_dict:", size_bytes, "bytes")
```

### Original Item 2 cell 28
```python
def tex_escape(s):
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#"), ("$", r"\$")]:
        s = str(s).replace(a, b)
    return s

results = {"env": ENV, "references": ref, "overfit": overfit, "lr_search": [{k: r[k] for k in ("lr", "best", "train_time_s", "steps", "peak_gpu_mem_mib")} for r in runs],
           "selected_lr": BEST_LR, "seeds": seed_summary, "val_report": report, "val_confusion": cm.tolist(),
           "cost": {**cost_static, "serialized_bytes": size_bytes, "cpu_bench": bench},
           "best_run_history": best_run["history"], "note": "Validation results only; the test set was not evaluated."}
json.dump(results, open(f"{OUT}/results/m0_baseline.json", "w"), indent=2, default=float)

c = counts; S = stats; R = removal_log
macros = {
  "NRowsRaw": S["rows"], "NTrainRaw": S["train"]["rows"], "NTestRaw": S["test"]["rows"],
  "NCVs": S["unique_cvs"], "NJobs": S["unique_jobs"], "NConflictPairs": S["conflicting_pairs"], "NConflictRows": S["conflicting_rows"],
  "NCVOverlap": S["overlap_train_test"]["cvs"], "NJobOverlap": S["overlap_train_test"]["jobs"],
  "NNearDupJobPairs": S["near_duplicates"]["job_pairs"], "NNearDupJobCross": S["near_duplicates"]["job_pairs_across_train_test"],
  "NNearDupCVPairs": S["near_duplicates"]["cv_pairs"],
  "NRemovedNearDup": R["removed_train_rows_near_duplicate_of_test_job"],
  "NTrain": c["train"]["rows"], "NVal": c["validation"]["rows"], "NTest": c["test"]["rows"],
  "NTrainJobs": c["train"]["jobs"], "NValJobs": c["validation"]["jobs"], "NTestJobs": c["test"]["jobs"],
  "NTrainNo": c["train"]["class_counts"]["No Fit"], "NTrainPot": c["train"]["class_counts"]["Potential Fit"], "NTrainGood": c["train"]["class_counts"]["Good Fit"],
  "NValNo": c["validation"]["class_counts"]["No Fit"], "NValPot": c["validation"]["class_counts"]["Potential Fit"], "NValGood": c["validation"]["class_counts"]["Good Fit"],
  "NTestNo": c["test"]["class_counts"]["No Fit"], "NTestPot": c["test"]["class_counts"]["Potential Fit"], "NTestGood": c["test"]["class_counts"]["Good Fit"],
  "TokPairMedian": f'{S["tokens"]["pair_median"]:.0f}', "TokPairNinety": f'{S["tokens"]["pair_p90"]:.0f}',
  "TruncPct": f'{S["tokens"]["pct_pairs_truncated_at_256"]:.1f}',
  "MajFone": f'{ref["majority"]["macro_f1"]:.1f}', "MajAcc": f'{ref["majority"]["accuracy"]:.1f}',
  "TfidfFone": f'{ref["tfidf_logreg"]["macro_f1"]:.1f}', "TfidfAcc": f'{ref["tfidf_logreg"]["accuracy"]:.1f}',
  "OverfitFirst": f'{overfit["first_loss"]:.3f}', "OverfitFinal": f'{overfit["final_loss"]:.4f}', "OverfitAcc": f'{overfit["final_accuracy"]:.0f}',
  "OverfitPassed": "passed" if overfit["passed"] else "did not pass",
  "MzeroLR": f"{BEST_LR:g}", "MzeroFone": f'{best_run["best"]["macro_f1"]:.1f}', "MzeroAcc": f'{best_run["best"]["accuracy"]:.1f}',
  "MzeroBestStep": best_run["best"]["step"], "MzeroSteps": best_run["steps"],
  "MzeroFoneMean": f'{seed_summary["macro_f1_mean"]:.1f}', "MzeroFoneSD": f'{seed_summary["macro_f1_sd"]:.1f}',
  "MzeroAccMean": f'{seed_summary["accuracy_mean"]:.1f}', "MzeroAccSD": f'{seed_summary["accuracy_sd"]:.1f}',
  "MzeroParams": f'{params["total"]:,}', "MzeroParamsM": f'{params["total"]/1e6:.2f}',
  "MzeroEmbParams": f'{params["embeddings"]:,}', "MzeroLayerParams": f'{params["encoder.layer.0"]:,}',
  "MzeroPoolerParams": f'{params["pooler"]:,}', "MzeroClsParams": f'{params["classifier"]:,}',
  "MzeroEmbShare": f'{100*params["embeddings"]/params["total"]:.1f}',
  "MzeroMACsM": f'{macs["total"]/1e6:.1f}', "MzeroLayerMACsM": f'{macs["encoder.layer.0"]/1e6:.1f}',
  "MzeroMACsAnalyticM": f'{macs_analytic["total"]/1e6:.1f}',
  "MzeroSizeMB": f'{size_bytes/2**20:.2f}', "MzeroLatMed": f'{bench["model_only"]["median_ms"]:.2f}',
  "MzeroLatPninetyfive": f'{bench["model_only"]["p95_ms"]:.2f}', "MzeroLatTokMed": f'{bench["tokenization_inclusive"]["median_ms"]:.2f}',
  "MzeroRSS": f'{bench["peak_rss_mib"]:.0f}', "MzeroRSSBase": f'{bench["rss_after_imports_mib"]:.0f}',
  "MzeroRSSDelta": f'{bench["peak_rss_mib"] - bench["rss_after_imports_mib"]:.0f}',
  "MzeroTrainTime": f'{best_run["train_time_s"]:.0f}',
  "MzeroPeakGPU": f'{best_run["peak_gpu_mem_mib"]:.0f}' if best_run["peak_gpu_mem_mib"] else "n/a",
  "EnvGPU": tex_escape(ENV["gpu"]), "EnvCPU": tex_escape(ENV["cpu"]), "EnvTorch": tex_escape(ENV["torch"]),
  "EnvTransformers": tex_escape(ENV["transformers"]),
  "StudentRev": ENV["student_revision"][:7],
}
with open(f"{OUT}/tex/report_numbers.tex", "w") as f:
    f.write("% Auto-generated by HireMind_M0_Baseline.ipynb - do not edit by hand\n")
    for k, v in macros.items():
        f.write(f"\\renewcommand{{\\{k}}}{{{v}}}\n")
lr_rows = " ".join(f"{r['lr']:g} & {r['best']['macro_f1']:.1f} & {r['best']['accuracy']:.1f} & {r['best']['loss']:.3f} & {r['best']['step']} \\\\" for r in runs)
with open(f"{OUT}/tex/report_numbers.tex", "a") as f:
    f.write("\\renewcommand{\\LRRows}{" + lr_rows + "}\n")
with open(f"{OUT}/tex/seed_rows.tex", "w") as f:
    for r in seed_runs:
        f.write(f"{r['seed']} & {r['best']['macro_f1']:.1f} & {r['best']['accuracy']:.1f} \\\\\n")
print(open(f"{OUT}/tex/report_numbers.tex").read())
```

### Original Item 2 cell 29
```python
shutil.make_archive("HireMind_A2_outputs", "zip", OUT)
print("Created HireMind_A2_outputs.zip (no raw text inside). Private files stay in ./private.")
try:
    from google.colab import files
    files.download("HireMind_A2_outputs.zip")
except Exception:
    print("Not running in Colab: download HireMind_A2_outputs.zip from the file browser.")
```

# Teacher recovery provenance — historical source, do not rerun

The following cells are retained verbatim from the uploaded executed notebook. The reconstructed weights and comparison are already saved. This appendix documents how they were produced; use reviewed_trial/R01 for present verification. Repeating these incident operations is not part of normal execution.

```python
import types
from pathlib import Path

assert torch.cuda.is_available(), "GPU required."
assert STORAGE_READY, "Storage verification required."

teacher_tag = "teacher_s42_lr5e-05"
public = Path(WORK) / "public"
recovery_output = public / "recovery_incident/teacher_reconstruction"
recovered_record = recovery_output / "teacher_recovered.json"
original_record = public / "trials" / f"{teacher_tag}.json"

# Confirm the original targets still belong to the original Teacher.
verified_teacher_targets(WORK, pin=True)

# Preserve the records and targets used by completed M1 experiments.
protected = [
    original_record,
    public / "teacher_selected.json",
    Path(WORK) / "private/teacher_train_logits.pt",
]
before = {str(path): sha(path) for path in protected}

original_completed_trial = completed_trial
original_frozen_write = frozen_write
original_dump = dump

def recovery_completed_trial(work, kind, lr, s, temperature=None):
    assert Path(work) == Path(WORK)
    assert (kind, lr, s, temperature) == ("teacher", 5e-5, 42, None)

    if recovered_record.exists():
        return validate_trial(
            work, read(recovered_record), expected_tag=teacher_tag
        )

    # Require a verified snapshot; never silently start from zero.
    assert recovery_candidates(work, teacher_tag), (
        "Teacher recovery checkpoint missing. Stop."
    )
    return None

def recovery_record_write(path, value):
    path = Path(path)
    if path == original_record:
        path = recovered_record
    return original_frozen_write(path, value)

def recovery_curve_write(path, value):
    path = Path(path)
    if (
        path.parent == public / "curves"
        and path.name.startswith(teacher_tag + "_")
    ):
        path = recovery_output / "curves" / path.name
    return original_dump(path, value)

# Use the existing training loop unchanged, with separate record routing.
# The original global train_trial function remains unchanged.
recovery_scope = dict(train_trial.__globals__)
recovery_scope.update({
    "completed_trial": recovery_completed_trial,
    "frozen_write": recovery_record_write,
    "dump": recovery_curve_write,
})
teacher_recovery_runner = types.FunctionType(
    train_trial.__code__,
    recovery_scope,
    name="teacher_recovery_runner",
    argdefs=train_trial.__defaults__,
    closure=train_trial.__closure__,
)

try:
    recovered_teacher = teacher_recovery_runner(
        WORK, "teacher", 5e-5, 42
    )
finally:
    for filename, expected_hash in before.items():
        assert sha(Path(filename)) == expected_hash, (
            f"Protected file changed: {filename}"
        )

print("RECOVERED TEACHER: completed weights validated.")
print("Weights:", recovered_teacher["weights"])
print("Separate result record:", recovered_record)
print("Original Teacher records and M1 targets preserved.")
print("Comparison with original Teacher evidence is still required.")
```

```python
from pathlib import Path
import torch
import gc

root = Path(WORK)
original = read(root / "public/teacher_selected.json")
recovered = read(
    root / "public/recovery_incident/"
    "teacher_reconstruction/teacher_recovered.json"
)
recovered = validate_trial(
    WORK, recovered, expected_tag="teacher_s42_lr5e-05"
)

verified_teacher_targets(WORK, pin=True)

same_file_hash = (
    recovered["weights_sha256"] == original["weights_sha256"]
)
print("Weights file matches original SHA-256:", same_file_hash)

if same_file_hash:
    print("PASS: recovered weights are byte-identical to the original.")
else:
    # Different serialization can change a file hash.
    # Compare predictions using the original evaluation procedure.
    assert torch.cuda.is_available(), "Keep the GPU connected for this check."

    cache = torch.load(
        root / "private/teacher_train_logits.pt",
        map_location="cpu",
        weights_only=True,
    )
    assert cache["training_row_ids"] == read(
        root / "public/data_used.json"
    )["training_row_ids"]

    teacher_model = load_trial(WORK, recovered).to("cuda")
    try:
        _, new_logits, _ = evaluate(
            teacher_model, tokens(WORK, "train"), "cuda",
            batch_size=16,
        )
        old_logits = cache["logits"].cpu()
        new_logits = new_logits.cpu()

        assert new_logits.shape == old_logits.shape
        assert torch.isfinite(new_logits).all()

        print("Logits exactly equal:",
              torch.equal(new_logits, old_logits))
        print("Maximum absolute difference:",
              (new_logits - old_logits).abs().max().item())
        print("Logits close (rtol=1e-5, atol=1e-6):",
              torch.allclose(
                  new_logits, old_logits, rtol=1e-5, atol=1e-6
              ))
        print("Predicted-class agreement (%):",
              100 * (
                  new_logits.argmax(-1) == old_logits.argmax(-1)
              ).float().mean().item())
    finally:
        del teacher_model
        gc.collect()
        torch.cuda.empty_cache()

print("Original records and cached predictions were not overwritten.")
```

```python
assert torch.equal(new_logits, cache["logits"].cpu())
assert recovered["steps"] == 939

evidence = {
    "teacher_tag": recovered["tag"],
    "original_weights_sha256": original["weights_sha256"],
    "recovered_weights": recovered["weights"],
    "recovered_weights_sha256": recovered["weights_sha256"],
    "cached_targets_sha256": sha(
        Path(WORK) / "private/teacher_train_logits.pt"
    ),
    "examples_checked": len(cache["training_row_ids"]),
    "scope": "Original cached training examples, in the verified row order.",
    "logits_exactly_equal": True,
    "maximum_absolute_difference": 0.0,
    "predicted_class_agreement_percent": 100.0,
    "byte_identical_weight_files": False,
    "original_targets_preserved": True,
    "verified_at": utc(),
}

evidence_path = (
    Path(WORK)
    / "public/recovery_incident/teacher_reconstruction"
    / "teacher_equivalence.json"
)

persist_recovery_metadata(evidence_path, evidence)
print("PASS: Teacher recovery comparison saved and verified.")
```


## Submission boundaries
- **A3:** updated research PDF and GitHub link with visible code, configs, validation results, tables and figures. Human audits and review must be completed before calling the author-review draft final.
- **A4:** freeze the final protocol before held-out testing, then complete the paper, presentation, poster, contribution portfolio and reproducibility package according to its separate rubric. F01–F03 remain disabled here.
- **Senior:** reuse validated model/tokenizer/preprocessing/label mapping and CPU inference only after deployment evaluation. Ranking, assessments, feedback and HR decisions stay separate. Do not automatically reject applicants.
- Keep large checkpoints and private data in controlled storage. A public evidence ZIP does not include the actual model weights and is not the full A4 submission.
